

# Memory Architecture for Agentic AI Systems: A Principal-Level Technical Report

## Formal Theory, Mathematical Foundations, Typed Protocols, and SOTA Algorithms for Production-Grade Agent Memory

---

## 1. Foundational Formalism: Memory as a Typed, Layered Control System

### 1.1 The Computational Model of Agent Memory

An agentic system operates under a fundamental constraint: the context window $W$ of size $|W| = N_{\max}$ tokens functions as a **bounded working register**, analogous to CPU registers plus L1 cache, not as unbounded RAM. Every token consumed by stale history, irrelevant retrieval, or uncompressed state is a token **denied** to reasoning, planning, and synthesis. Memory architecture is therefore a **resource-allocation optimization problem** under a hard token budget.

**Definition 1 (Agent Memory System).** An agent memory system is a tuple:

$$\mathcal{M} = \langle W, S, E, \Sigma, P, \phi_{\text{admit}}, \phi_{\text{evict}}, \phi_{\text{retrieve}}, \phi_{\text{promote}}, \phi_{\text{consolidate}} \rangle$$

where:

| Symbol | Type | Description |
|--------|------|-------------|
| $W$ | `BoundedBuffer<Token>` | Working memory (active context window) |
| $S$ | `SessionStore<MemoryItem>` | Session-scoped ephemeral memory |
| $E$ | `DurableStore<EpisodicRecord>` | Validated episodic memory (interaction traces) |
| $\Sigma$ | `DurableStore<SemanticFact>` | Canonical semantic/organizational knowledge |
| $P$ | `DurableStore<Procedure>` | Procedural memory (learned workflows) |
| $\phi_{\text{admit}}$ | `AdmissionPolicy` | Gate function controlling writes |
| $\phi_{\text{evict}}$ | `EvictionPolicy` | Eviction/pruning control |
| $\phi_{\text{retrieve}}$ | `RetrievalEngine` | Multi-signal ranked retrieval |
| $\phi_{\text{promote}}$ | `PromotionPolicy` | Layer-crossing promotion logic |
| $\phi_{\text{consolidate}}$ | `ConsolidationEngine` | Merge, deduplicate, compress |

**Definition 2 (Memory Item).** Every item stored in any layer is a provenance-tagged record:

$$m = \langle \text{id}, \text{content}, \text{embedding}, \text{source}, \text{timestamp}, \text{ttl}, \text{importance}, \text{access\_count}, \text{lineage}, \text{version}, \text{schema\_type} \rangle$$

```protobuf
message MemoryItem {
  string id = 1;
  string content = 2;
  repeated float embedding = 3;
  Provenance source = 4;
  google.protobuf.Timestamp created_at = 5;
  google.protobuf.Duration ttl = 6;
  float importance_score = 7;
  uint64 access_count = 8;
  repeated string lineage_ids = 9;
  uint32 version = 10;
  MemorySchemaType schema_type = 11;
  
  enum MemorySchemaType {
    EPISODIC = 0;
    SEMANTIC = 1;
    PROCEDURAL = 2;
    WORKING = 3;
    SESSION = 4;
  }
}

message Provenance {
  string origin_agent_id = 1;
  string task_id = 2;
  string tool_invocation_id = 3;
  string human_annotator_id = 4;
  float confidence = 5;
  ValidationStatus validation = 6;
}
```

### 1.2 The Token Budget Constraint

At every agent step $t$, the working memory $W_t$ must satisfy:

$$|W_t| = |\text{RolePolicy}| + |\text{TaskState}_t| + |\text{RetrievedEvidence}_t| + |\text{ToolAffordances}_t| + |\text{MemorySummary}_t| + |\text{History}_t| \leq N_{\max} - R_{\text{reserved}}$$

where $R_{\text{reserved}}$ is a **hard reservation** for generation output and chain-of-thought reasoning. Violating this constraint degrades generation quality non-linearly:

$$Q(\text{generation}) \propto \begin{cases} 1 & \text{if } |W_t| \leq N_{\max} - R_{\text{reserved}} \\ e^{-\lambda(|W_t| - (N_{\max} - R_{\text{reserved}}))} & \text{otherwise} \end{cases}$$

where $\lambda > 0$ is a model-specific degradation coefficient empirically measured via eval suites. The objective function for context construction at each step is:

$$\max_{C \subseteq \mathcal{C}_{\text{candidates}}} \sum_{c \in C} u(c, \text{task}_t) \quad \text{subject to} \quad \sum_{c \in C} \text{tokens}(c) \leq B_t$$

where $u(c, \text{task}_t)$ is the **task-conditioned utility** of context item $c$, and $B_t = N_{\max} - R_{\text{reserved}} - |\text{FixedPrefill}|$ is the available budget at step $t$.

> **This is a variant of the 0/1 Knapsack Problem** and is NP-hard in general. SOTA systems approximate it via greedy utility-density ranking or learned value functions.

---

## 2. Working Memory: The Active Reasoning Register

### 2.1 Formal Specification

Working memory $W$ is the **only memory layer visible to the model at inference time**. It is compiled—not appended—at each agent step by the **Prefill Compiler**.

**Definition 3 (Prefill Compiler).** A deterministic function:

$$\text{Compile}: (\text{RolePolicy}, \text{TaskState}_t, \text{Retrieved}_t, \text{Tools}_t, \text{MemSummary}_t, \text{History}_t) \rightarrow W_t$$

subject to $|W_t| \leq B_t$.

### 2.2 SOTA Context Compression: Hierarchical Summarization with Importance Weighting

Rather than naive truncation (dropping the oldest $k$ messages), SOTA systems apply **hierarchical importance-weighted summarization**.

**Definition 4 (Importance-Weighted Message Retention).** For a message history $H = [h_1, h_2, \ldots, h_n]$, assign each message an importance score:

$$I(h_i) = \alpha \cdot \text{Recency}(h_i) + \beta \cdot \text{TaskRelevance}(h_i, \text{task}_t) + \gamma \cdot \text{InformationDensity}(h_i) + \delta \cdot \text{ReferenceCount}(h_i)$$

where:

$$\text{Recency}(h_i) = e^{-\mu(t - t_i)}$$

$$\text{TaskRelevance}(h_i, \text{task}_t) = \cos(\mathbf{e}_{h_i}, \mathbf{e}_{\text{task}_t})$$

$$\text{InformationDensity}(h_i) = \frac{|\text{NamedEntities}(h_i)| + |\text{ToolCalls}(h_i)| + |\text{Decisions}(h_i)|}{|\text{tokens}(h_i)|}$$

$$\text{ReferenceCount}(h_i) = \sum_{j > i} \mathbb{1}[h_j \text{ references } h_i]$$

and $\alpha + \beta + \gamma + \delta = 1$ are tunable weights.

### 2.3 Algorithm: Adaptive Context Window Management

```
ALGORITHM 1: AdaptiveContextWindowManager
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:
  H[1..n]          — full message history
  task_t            — current task descriptor
  B_t               — available token budget
  R_reserved        — reserved reasoning tokens
  fixed_prefill     — compiled role policy + tool affordances

Output:
  W_t               — compiled working memory (context window payload)

PROCEDURE:
  1.  budget_remaining ← B_t - tokens(fixed_prefill)
  
  2.  // PHASE 1: Score all history items
      FOR each h_i IN H[1..n]:
          I(h_i) ← α·Recency(h_i) + β·TaskRelevance(h_i, task_t)
                    + γ·InformationDensity(h_i) + δ·ReferenceCount(h_i)
      END FOR
  
  3.  // PHASE 2: Partition into tiers
      tier_critical ← {h_i : I(h_i) ≥ θ_critical}      // Must retain verbatim
      tier_important ← {h_i : θ_important ≤ I(h_i) < θ_critical}  // Retain or summarize
      tier_background ← {h_i : I(h_i) < θ_important}    // Summarize or discard
  
  4.  // PHASE 3: Greedy knapsack packing
      W_t ← fixed_prefill
      
      // Pack critical items first (verbatim)
      FOR each h_i IN tier_critical SORTED BY I(h_i) DESC:
          IF tokens(h_i) ≤ budget_remaining:
              APPEND h_i TO W_t
              budget_remaining -= tokens(h_i)
          ELSE:
              summary_i ← CompressMessage(h_i, target_ratio=0.3)
              IF tokens(summary_i) ≤ budget_remaining:
                  APPEND summary_i TO W_t
                  budget_remaining -= tokens(summary_i)
      END FOR
      
      // Pack important items (summarize if needed)
      FOR each h_i IN tier_important SORTED BY I(h_i) DESC:
          IF tokens(h_i) ≤ budget_remaining:
              APPEND h_i TO W_t
              budget_remaining -= tokens(h_i)
          ELSE IF budget_remaining > MIN_SUMMARY_TOKENS:
              summary_i ← CompressMessage(h_i, target_ratio=0.2)
              APPEND summary_i TO W_t
              budget_remaining -= tokens(summary_i)
      END FOR
      
      // Pack background as batch summary
      IF |tier_background| > 0 AND budget_remaining > MIN_BATCH_SUMMARY:
          batch_summary ← BatchSummarize(tier_background, max_tokens=budget_remaining)
          APPEND batch_summary TO W_t
      END IF
  
  5.  // PHASE 4: Inject retrieved evidence and memory summaries
      retrieved ← φ_retrieve(task_t, budget=budget_remaining × 0.6)
      mem_summary ← CompileMemorySummary(S, E, Σ, budget=budget_remaining × 0.4)
      APPEND retrieved TO W_t
      APPEND mem_summary TO W_t
  
  6.  ASSERT tokens(W_t) ≤ N_max - R_reserved
      RETURN W_t
```

### 2.4 Compression Functions: SOTA Approaches

**Extractive Compression** selects salient sentences. **Abstractive Compression** generates condensed summaries. The SOTA approach is **LLM-as-Compressor with Fidelity Verification**:

$$\text{CompressMessage}(h, r) = \text{LLM}_{\text{compress}}\left(\text{prompt}_{\text{compress}}, h, \lfloor |h| \cdot r \rfloor\right)$$

with a fidelity verification step:

$$\text{Fidelity}(h, \hat{h}) = \frac{|\text{Facts}(h) \cap \text{Facts}(\hat{h})|}{|\text{Facts}(h)|}$$

Only accept $\hat{h}$ if $\text{Fidelity}(h, \hat{h}) \geq \tau_{\text{fidelity}}$ (typically $\tau_{\text{fidelity}} \geq 0.95$).

---

## 3. Session Memory: Ephemeral Task-Scoped State

### 3.1 Formal Definition

Session memory $S$ persists for the duration of a **single user session or task execution** and is destroyed on session termination. It serves as the **scratchpad** for multi-step task state.

$$S = \{s_k : s_k = \langle \text{task\_id}, \text{key}, \text{value}, \text{step\_created}, \text{step\_last\_accessed} \rangle\}$$

### 3.2 Task State Storage and Recall Protocol

```
ALGORITHM 2: TaskStateScratchpad
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRUCTURE TaskScratchpad:
    store: HashMap<TaskID, HashMap<String, TypedValue>>
    max_entries_per_task: uint = 256
    ttl_per_entry: Duration = session_duration

PROCEDURE OffloadToScratchpad(task_id, key, value, step_t):
    // Offload from working memory to session scratchpad
    1. VALIDATE Schema(value) against expected_type(key)
    2. IF store[task_id].size ≥ max_entries_per_task:
           EVICT entry with MIN(step_last_accessed) from store[task_id]
    3. store[task_id][key] ← {value, step_created=step_t, step_last_accessed=step_t}
    4. EMIT Trace("scratchpad.write", task_id, key, tokens(value))

PROCEDURE RecallFromScratchpad(task_id, keys[], step_t):
    // Recall specific keys back into working memory
    1. results ← []
    2. FOR each k IN keys:
           IF k IN store[task_id]:
               store[task_id][k].step_last_accessed ← step_t
               APPEND store[task_id][k].value TO results
           ELSE:
               APPEND NULL TO results
               EMIT Warning("scratchpad.miss", task_id, k)
    3. RETURN results

PROCEDURE FlushOnSessionEnd(task_id):
    // Evaluate each entry for promotion to episodic memory
    1. FOR each (key, entry) IN store[task_id]:
           candidate ← ConstructMemoryItem(entry)
           IF φ_admit(candidate) = ACCEPT:
               φ_promote(candidate, target=E)
    2. DELETE store[task_id]
    3. EMIT Trace("scratchpad.flush", task_id)
```

### 3.3 Mathematical Model of Scratchpad Utility

The utility of offloading item $x$ from $W$ to $S$ at step $t$ is:

$$U_{\text{offload}}(x, t) = \underbrace{\text{tokens}(x)}_{\text{freed capacity}} \times \underbrace{(1 - P(\text{need } x \text{ at step } t+1))}_{\text{probability of non-use next step}}$$

Offload when $U_{\text{offload}}(x, t) > \theta_{\text{offload}}$. The probability of need is estimated via:

$$P(\text{need } x \text{ at step } t+1) = \sigma\left(w_1 \cdot \text{TaskRelevance}(x) + w_2 \cdot \text{StepProximity}(x) + w_3 \cdot \text{DependencyCount}(x)\right)$$

where $\sigma$ is the sigmoid function and $w_i$ are learned or heuristic weights.

---

## 4. Episodic Memory: Validated Interaction Traces

### 4.1 Formal Specification

Episodic memory $E$ stores **specific, validated interaction episodes** with full provenance. Each episode is a structured trace of a completed agent interaction.

$$e = \langle \text{episode\_id}, \text{task\_type}, \text{input\_summary}, \text{action\_trace}, \text{outcome}, \text{reward\_signal}, \text{timestamp}, \text{provenance}, \text{embeddings} \rangle$$

```protobuf
message EpisodicRecord {
  string episode_id = 1;
  string task_type = 2;
  string input_summary = 3;
  repeated ActionStep action_trace = 4;
  Outcome outcome = 5;
  float reward_signal = 6;
  google.protobuf.Timestamp timestamp = 7;
  Provenance provenance = 8;
  repeated float summary_embedding = 9;
  repeated float outcome_embedding = 10;
  uint64 retrieval_count = 11;
  float decayed_importance = 12;
}

message ActionStep {
  uint32 step_index = 1;
  string action_type = 2;       // "tool_call", "reasoning", "retrieval", "user_interaction"
  string action_detail = 3;
  string observation = 4;
  float step_confidence = 5;
}

message Outcome {
  bool success = 1;
  string result_summary = 2;
  repeated string error_codes = 3;
  string correction_applied = 4;
}
```

### 4.2 Episodic Importance Scoring with Temporal Decay

The importance of an episodic memory decays over time but is reinforced by retrieval:

$$I_E(e, t) = I_0(e) \cdot e^{-\lambda_{\text{decay}}(t - t_e)} + \eta \sum_{r \in \text{retrievals}(e)} e^{-\lambda_{\text{decay}}(t - t_r)}$$

where:
- $I_0(e)$ is the initial importance score assigned at admission
- $\lambda_{\text{decay}}$ is the temporal decay rate
- $\eta$ is the reinforcement coefficient per retrieval event
- $t_r$ are timestamps of past retrievals of this episode

This follows the **Ebbinghaus forgetting curve** augmented with **spaced-retrieval reinforcement**, a principle from cognitive science that SOTA memory systems operationalize.

### 4.3 Algorithm: Episodic Memory Admission Controller

```
ALGORITHM 3: EpisodicAdmissionController
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:
  candidate       — candidate memory item from completed interaction
  E               — current episodic store
  config          — admission thresholds and policies

Output:
  ACCEPT or REJECT decision with reason

PROCEDURE AdmitToEpisodicMemory(candidate, E, config):

  // GATE 1: Schema Validation
  1. IF NOT ValidateSchema(candidate, EpisodicRecord.schema):
         RETURN REJECT("schema_violation")

  // GATE 2: Novelty Check (prevent redundant storage)
  2. nearest_neighbors ← SemanticSearch(E, candidate.summary_embedding, k=5)
     max_similarity ← MAX({cos(candidate.summary_embedding, nn.summary_embedding)
                           : nn ∈ nearest_neighbors})
     IF max_similarity > config.dedup_threshold:  // typically 0.92
         // Check if candidate has novel outcome information
         outcome_sim ← cos(candidate.outcome_embedding,
                           nearest_neighbors[0].outcome_embedding)
         IF outcome_sim > config.outcome_dedup_threshold:  // typically 0.88
             RETURN REJECT("duplicate_episode")
         ELSE:
             // Merge: update existing episode with new outcome data
             MergeEpisode(nearest_neighbors[0], candidate)
             RETURN ACCEPT("merged_with_existing")

  // GATE 3: Importance Scoring via LLM Reflection
  3. importance_prompt ← ConstructReflectionPrompt(candidate)
     reflection ← LLM_evaluate(importance_prompt)
     I_0 ← ParseImportanceScore(reflection)  // ∈ [0.0, 1.0]
     
     IF I_0 < config.min_importance_threshold:  // typically 0.3
         RETURN REJECT("low_importance")

  // GATE 4: Factual Consistency Verification
  4. IF candidate.outcome.success = TRUE:
         consistency ← VerifyFactualConsistency(candidate.action_trace,
                                                  candidate.outcome)
         IF consistency < config.consistency_threshold:  // typically 0.85
             FLAG candidate AS "requires_human_review"

  // GATE 5: Provenance Completeness
  5. IF candidate.provenance.confidence < config.min_provenance_confidence:
         RETURN REJECT("insufficient_provenance")

  // COMMIT
  6. candidate.decayed_importance ← I_0
     candidate.retrieval_count ← 0
     WRITE candidate TO E with IDEMPOTENCY_KEY = candidate.episode_id
     EMIT Trace("episodic.admit", candidate.episode_id, I_0)
     RETURN ACCEPT("committed")
```

### 4.4 LLM Reflection for Importance Scoring

The importance score $I_0(e)$ is computed via a **structured LLM reflection call**, not a heuristic:

$$I_0(e) = \text{LLM}_{\text{judge}}\left(\text{prompt}_{\text{reflect}}, e\right) \rightarrow [0, 1]$$

The reflection prompt enforces a multi-dimensional rubric:

$$I_0(e) = \frac{1}{|\mathcal{D}|} \sum_{d \in \mathcal{D}} \text{score}_d(e)$$

where $\mathcal{D} = \{\text{novelty}, \text{correctability}, \text{generalizability}, \text{failure\_informativeness}, \text{user\_preference\_signal}\}$.

Each dimension is scored on $[0, 1]$. The composite score determines admission. **Only non-obvious corrections, constraint discoveries, and preference signals that would alter future behavior pass the gate.** Routine successful completions with no novel information are rejected to prevent memory bloat.

---

## 5. Semantic Memory: Canonical Organizational Knowledge

### 5.1 Formal Specification

Semantic memory $\Sigma$ stores **validated, general-purpose facts, rules, and domain knowledge** independent of specific interaction episodes.

$$\sigma = \langle \text{fact\_id}, \text{assertion}, \text{domain}, \text{confidence}, \text{source\_episodes}[], \text{contradicts}[], \text{last\_validated}, \text{embedding} \rangle$$

### 5.2 Promotion from Episodic to Semantic Memory

A fact is promoted from episodic to semantic memory when it is **observed consistently across multiple independent episodes**:

$$P(\text{promote } f \text{ to } \Sigma) = \sigma\left(\sum_{e \in E_f} w_e \cdot \text{confidence}(f, e) - \theta_{\text{promote}}\right)$$

where $E_f = \{e \in E : f \text{ is attested in } e\}$ and $w_e = I_E(e, t)$.

```
ALGORITHM 4: SemanticPromotionEngine
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:
  E              — episodic memory store
  Σ              — semantic memory store
  config         — promotion thresholds

PROCEDURE RunPromotionCycle():

  // STEP 1: Extract candidate facts from recent episodes
  1. recent_episodes ← E.query(created_after=now()-config.promotion_window)
     candidate_facts ← []
     FOR each e IN recent_episodes:
         facts ← ExtractFactualAssertions(e)  // NER + relation extraction
         FOR each f IN facts:
             candidate_facts.append((f, e))

  // STEP 2: Cluster identical/near-identical facts
  2. fact_clusters ← SemanticCluster(candidate_facts,
                                       threshold=config.fact_similarity_threshold)

  // STEP 3: Evaluate promotion criteria per cluster
  3. FOR each cluster IN fact_clusters:
         attestation_count ← |cluster.source_episodes|
         mean_confidence ← MEAN({confidence(f, e) : (f, e) ∈ cluster})
         source_diversity ← |UNIQUE(e.provenance.origin_agent_id : e ∈ cluster)|
         
         promotion_score ← (
             config.w_attestation × min(attestation_count / config.required_attestations, 1.0)
           + config.w_confidence × mean_confidence
           + config.w_diversity × min(source_diversity / config.required_sources, 1.0)
         )
         
         IF promotion_score ≥ config.promotion_threshold:
             // Check for contradictions with existing semantic memory
             contradictions ← FindContradictions(cluster.canonical_fact, Σ)
             IF |contradictions| > 0:
                 ESCALATE_TO_HUMAN_REVIEW(cluster, contradictions)
             ELSE:
                 σ_new ← ConstructSemanticFact(cluster)
                 WRITE σ_new TO Σ
                 EMIT Trace("semantic.promote", σ_new.fact_id, promotion_score)
```

### 5.3 Contradiction Detection

When a candidate fact $f_{\text{new}}$ potentially contradicts an existing fact $f_{\text{existing}} \in \Sigma$:

$$\text{Contradiction}(f_{\text{new}}, f_{\text{existing}}) = \begin{cases} 1 & \text{if } \text{NLI}(f_{\text{new}}, f_{\text{existing}}) = \texttt{CONTRADICTION} \wedge \cos(\mathbf{e}_{f_{\text{new}}}, \mathbf{e}_{f_{\text{existing}}}) > \theta_{\text{topic}} \\ 0 & \text{otherwise} \end{cases}$$

where $\text{NLI}$ is a **Natural Language Inference** model (e.g., DeBERTa-v3-large fine-tuned on MNLI). The topic similarity gate prevents false positives from unrelated domains.

---

## 6. Procedural Memory: Learned Workflow Internalization

### 6.1 Formal Specification

Procedural memory $P$ stores **reusable action sequences** (workflows, strategies, routines) extracted from successful episodic traces.

$$p = \langle \text{procedure\_id}, \text{trigger\_condition}, \text{action\_sequence}[], \text{preconditions}, \text{postconditions}, \text{success\_rate}, \text{execution\_count}, \text{avg\_latency}, \text{source\_episodes}[] \rangle$$

```protobuf
message Procedure {
  string procedure_id = 1;
  string trigger_description = 2;
  repeated float trigger_embedding = 3;
  repeated ProcedureStep steps = 4;
  repeated string preconditions = 5;
  repeated string postconditions = 6;
  float success_rate = 7;
  uint64 execution_count = 8;
  float avg_latency_ms = 9;
  repeated string source_episode_ids = 10;
  uint32 version = 11;
}

message ProcedureStep {
  uint32 order = 1;
  string action_type = 2;
  string action_template = 3;
  repeated string required_inputs = 4;
  repeated string expected_outputs = 5;
  string fallback_action = 6;
}
```

### 6.2 Algorithm: Procedural Memory Extraction via Trace Alignment

```
ALGORITHM 5: ProceduralMemoryExtractor
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:
  E                — episodic memory store
  P                — procedural memory store
  config           — extraction thresholds

PROCEDURE ExtractProcedures():

  // STEP 1: Identify recurring successful task patterns
  1. successful_episodes ← E.query(outcome.success=TRUE,
                                     created_after=now()-config.extraction_window)
     task_type_groups ← GroupBy(successful_episodes, key=task_type)

  // STEP 2: For each task type, align action traces
  2. FOR each (task_type, episodes) IN task_type_groups:
         IF |episodes| < config.min_episodes_for_extraction:  // typically ≥ 3
             CONTINUE
         
         // Extract action trace sequences
         traces ← [e.action_trace FOR e IN episodes]
         
         // Multiple Sequence Alignment (MSA) on action types
         // Using Needleman-Wunsch adapted for action sequences
         aligned ← MultipleSequenceAlignment(traces,
                       similarity_fn=ActionSimilarity,
                       gap_penalty=config.gap_penalty)
         
         // Identify conserved (consensus) steps
         consensus_steps ← []
         FOR each position IN aligned.columns:
             actions_at_pos ← aligned.column(position)
             conservation ← MostFrequent(actions_at_pos).frequency / |actions_at_pos|
             IF conservation ≥ config.conservation_threshold:  // typically ≥ 0.7
                 consensus_step ← Generalize(MostFrequent(actions_at_pos))
                 consensus_steps.append(consensus_step)

  // STEP 3: Construct procedure record
  3.     IF |consensus_steps| ≥ config.min_procedure_length:
             trigger ← SynthesizeTrigger(task_type, episodes)
             preconditions ← ExtractPreconditions(episodes)
             postconditions ← ExtractPostconditions(episodes)
             
             p_new ← Procedure {
                 procedure_id = GenerateID(),
                 trigger_description = trigger,
                 trigger_embedding = Embed(trigger),
                 steps = consensus_steps,
                 preconditions = preconditions,
                 postconditions = postconditions,
                 success_rate = MEAN({e.reward_signal : e ∈ episodes}),
                 execution_count = 0,
                 source_episode_ids = [e.episode_id FOR e IN episodes]
             }
             
             // Deduplicate against existing procedures
             existing_match ← P.query(
                 cos(trigger_embedding, p_new.trigger_embedding) > 0.90
             )
             IF existing_match:
                 MergeProcedure(existing_match, p_new)
             ELSE:
                 WRITE p_new TO P
                 EMIT Trace("procedural.extract", p_new.procedure_id)
```

### 6.3 Action Sequence Similarity (Needleman-Wunsch Adaptation)

The similarity between two action steps $a_i, a_j$ is:

$$\text{ActionSimilarity}(a_i, a_j) = \omega_1 \cdot \mathbb{1}[a_i.\text{type} = a_j.\text{type}] + \omega_2 \cdot \cos(\mathbf{e}_{a_i.\text{detail}}, \mathbf{e}_{a_j.\text{detail}}) + \omega_3 \cdot \text{IOOverlap}(a_i, a_j)$$

where:

$$\text{IOOverlap}(a_i, a_j) = \frac{|a_i.\text{inputs} \cap a_j.\text{inputs}| + |a_i.\text{outputs} \cap a_j.\text{outputs}|}{|a_i.\text{inputs} \cup a_j.\text{inputs}| + |a_i.\text{outputs} \cup a_j.\text{outputs}|}$$

---

## 7. Memory Retrieval Engine: Multi-Signal Ranked Retrieval

### 7.1 Architecture: Hybrid Retrieval Pipeline

Retrieval is the **critical path** that determines whether memory is useful. SOTA retrieval combines multiple signals in a unified ranking framework.

$$\phi_{\text{retrieve}}(\text{query}, \text{budget}) \rightarrow \text{RankedResults}[\text{MemoryItem}]$$

### 7.2 Multi-Signal Scoring Function

For each candidate memory item $m$ and query $q$:

$$\text{Score}(m, q) = \sum_{s \in \mathcal{S}} w_s \cdot f_s(m, q)$$

where $\mathcal{S}$ is the set of scoring signals:

| Signal $s$ | Function $f_s(m, q)$ | Description |
|---|---|---|
| Semantic | $\cos(\mathbf{e}_m, \mathbf{e}_q)$ | Dense vector similarity |
| Lexical | $\text{BM25}(m.\text{content}, q.\text{terms})$ | Exact term matching |
| Recency | $e^{-\lambda(t_{\text{now}} - m.\text{timestamp})}$ | Temporal decay |
| Authority | $m.\text{provenance.confidence} \times \text{SourceTrust}(m.\text{source})$ | Provenance-weighted trust |
| Freshness | $\mathbb{1}[m.\text{version} = \text{latest}(m.\text{fact\_id})]$ | Version currency |
| Task Affinity | $\cos(\mathbf{e}_m, \mathbf{e}_{\text{task\_type}})$ | Relevance to current task type |
| Access Pattern | $\log(1 + m.\text{access\_count}) \times \text{Recency}_{\text{access}}$ | Popularity with recency |
| Graph Proximity | $\text{PPR}(m, q, \mathcal{G}_{\text{knowledge}})$ | Personalized PageRank in knowledge graph |

### 7.3 Query Decomposition and Routing

Before retrieval, the query is **decomposed, expanded, and routed** per signal source:

```
ALGORITHM 6: QueryDecompositionAndRouting
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:
  raw_query        — original user/agent query
  memory_layers    — {W, S, E, Σ, P}
  latency_budget   — maximum retrieval latency (ms)

Output:
  ranked_results   — provenance-tagged, ranked memory items

PROCEDURE RetrieveWithDecomposition(raw_query, memory_layers, latency_budget):

  // STEP 1: Query Analysis and Decomposition
  1. analysis ← LLM_analyze_query(raw_query)
     subqueries ← analysis.decomposed_subqueries     // e.g., ["user travel preferences",
                                                      //        "airline booking workflow",
                                                      //        "visa requirements for Japan"]
     query_type ← analysis.type                       // factual | procedural | episodic | hybrid
     temporal_scope ← analysis.temporal_scope         // recent | historical | all

  // STEP 2: Route subqueries to appropriate memory layers and retrieval modes
  2. retrieval_plan ← []
     FOR each sq IN subqueries:
         targets ← RouteToLayers(sq, query_type, temporal_scope)
         // Example routing:
         //   "user travel preferences" → E (episodic) + Σ (semantic), mode=semantic_search
         //   "airline booking workflow" → P (procedural), mode=trigger_match
         //   "visa requirements" → Σ (semantic), mode=hybrid(BM25 + semantic)
         FOR each (layer, mode) IN targets:
             retrieval_plan.append({
                 subquery: sq,
                 expanded_query: ExpandQuery(sq),  // synonym expansion, HyDE
                 layer: layer,
                 mode: mode,
                 deadline: latency_budget / |retrieval_plan|
             })

  // STEP 3: Parallel Execution with Deadline Enforcement
  3. raw_results ← ParallelExecute(retrieval_plan,
                                     timeout=latency_budget,
                                     strategy=RETURN_AVAILABLE_ON_TIMEOUT)

  // STEP 4: Cross-Layer Fusion and Reranking
  4. fused ← DeduplicateAndMerge(raw_results)
     
     // Multi-signal scoring
     FOR each m IN fused:
         m.final_score ← Σ_s (w_s × f_s(m, raw_query))
     
     // LLM-based reranking on top-K candidates
     top_k ← TopK(fused, k=config.rerank_pool_size)  // typically k=20
     reranked ← LLM_rerank(top_k, raw_query,
                             criteria=["relevance", "actionability", "recency"])

  // STEP 5: Token-Budget-Aware Truncation
  5. selected ← []
     budget_remaining ← config.retrieval_token_budget
     FOR each m IN reranked:
         IF tokens(m.content) ≤ budget_remaining:
             selected.append(m)
             budget_remaining -= tokens(m.content)
         ELSE IF budget_remaining > MIN_ITEM_TOKENS:
             compressed ← CompressForContext(m, max_tokens=budget_remaining)
             selected.append(compressed)
             BREAK
     
     // Attach provenance tags
     FOR each m IN selected:
         m.provenance_tag ← FormatProvenance(m.source, m.confidence, m.timestamp)
     
     RETURN selected
```

### 7.4 Hypothetical Document Embedding (HyDE) for Query Expansion

Instead of embedding the raw query directly, generate a **hypothetical answer** and embed that:

$$\mathbf{e}_{\text{HyDE}} = \text{Embed}\left(\text{LLM}_{\text{generate}}(\text{"Answer this as if you knew: "} + q)\right)$$

$$\text{Retrieval}(q) = \text{TopK}\left(\{m \in \mathcal{M} : \cos(\mathbf{e}_m, \mathbf{e}_{\text{HyDE}}) > \theta\}\right)$$

This bridges the vocabulary gap between queries and stored documents. The SOTA enhancement is **multi-HyDE**: generate $k$ hypothetical documents from different perspectives and average their embeddings:

$$\mathbf{e}_{\text{multi-HyDE}} = \frac{1}{k}\sum_{i=1}^{k} \text{Embed}\left(\text{LLM}_{\text{generate}}(q, \text{perspective}_i)\right)$$

### 7.5 Iterative Retrieval with Self-Critique

```
ALGORITHM 7: IterativeRetrievalWithCritique
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:
  query            — original query
  max_iterations   — bounded iteration count (typically 3)
  quality_threshold — minimum retrieval quality score

Output:
  final_results    — high-quality retrieved items

PROCEDURE IterativeRetrieve(query, max_iterations, quality_threshold):
  
  current_query ← query
  accumulated_results ← []
  
  FOR iteration IN 1..max_iterations:
      // Retrieve
      results_i ← φ_retrieve(current_query, budget)
      accumulated_results ← Merge(accumulated_results, results_i)
      
      // Critique: does the retrieved set answer the query?
      critique ← LLM_critique(query, accumulated_results)
      // critique = {quality_score: float, gaps: [str], refinement_suggestion: str}
      
      IF critique.quality_score ≥ quality_threshold:
          BREAK  // Sufficient quality achieved
      
      IF |critique.gaps| = 0:
          BREAK  // No identifiable gaps; further iteration unlikely to help
      
      // Refine query based on identified gaps
      current_query ← LLM_refine_query(query, critique.gaps,
                                          critique.refinement_suggestion,
                                          already_retrieved=accumulated_results)
  
  RETURN accumulated_results
```

---

## 8. Memory Consolidation and Pruning Engine

### 8.1 The Consolidation Problem

Over time, memory stores accumulate **redundant, stale, contradictory, and low-value entries**. Consolidation is the process of maintaining memory health, analogous to garbage collection in runtime systems.

### 8.2 Formal Pruning Criteria

A memory item $m$ is a **pruning candidate** if:

$$\text{PruneScore}(m, t) = \underbrace{w_{\text{age}} \cdot \frac{t - m.\text{timestamp}}{T_{\max}}}_{\text{staleness}} + \underbrace{w_{\text{access}} \cdot \left(1 - \frac{\log(1 + m.\text{access\_count})}{\log(1 + A_{\max})}\right)}_{\text{disuse}} + \underbrace{w_{\text{importance}} \cdot (1 - I_E(m, t))}_{\text{low importance}} + \underbrace{w_{\text{redundancy}} \cdot R(m, \mathcal{M})}_{\text{redundancy with peers}}$$

where the redundancy score is:

$$R(m, \mathcal{M}) = \max_{m' \in \mathcal{M} \setminus \{m\}} \cos(\mathbf{e}_m, \mathbf{e}_{m'}) \times \mathbb{1}[m'.\text{importance} \geq m.\text{importance}]$$

Prune $m$ if $\text{PruneScore}(m, t) > \theta_{\text{prune}}$.

### 8.3 Algorithm: Memory Consolidation Agent

```
ALGORITHM 8: MemoryConsolidationAgent
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:
  memory_layers   — {E, Σ, P}
  config          — consolidation policies

// Runs as a scheduled background agent (e.g., every 6 hours)
PROCEDURE RunConsolidation(memory_layers, config):

  FOR each layer IN memory_layers:
      all_items ← layer.scan()
      
      // PHASE 1: Compute prune scores
      prune_candidates ← []
      FOR each m IN all_items:
          ps ← ComputePruneScore(m, now(), config)
          IF ps > config.prune_threshold:
              prune_candidates.append((m, ps))
      
      // PHASE 2: Sort by prune score descending (most pruneable first)
      SORT prune_candidates BY ps DESC
      
      // PHASE 3: Execute pruning with safety checks
      pruned_count ← 0
      FOR each (m, ps) IN prune_candidates:
          IF pruned_count ≥ config.max_prune_per_cycle:
              BREAK  // Rate limit to prevent mass deletion
          
          // Safety: check if any active procedure or task references this item
          IF IsReferencedByActiveProcedure(m, P) OR IsReferencedByActiveTask(m):
              CONTINUE  // Skip: item is in active use
          
          // For items above high-confidence prune threshold, auto-delete
          IF ps > config.auto_prune_threshold:
              ArchiveAndDelete(m, layer)  // Soft-delete with archive
              pruned_count += 1
          ELSE:
              // Items in uncertain zone: merge or summarize instead of delete
              merge_target ← FindMergeCandidate(m, layer)
              IF merge_target ≠ NULL:
                  MergeItems(merge_target, m, layer)
                  pruned_count += 1
      
      // PHASE 4: Deduplication pass
      embeddings ← [m.embedding FOR m IN layer.scan()]
      duplicate_pairs ← FindNearDuplicates(embeddings,
                                              threshold=config.dedup_cos_threshold)
      FOR each (m_a, m_b) IN duplicate_pairs:
          survivor ← SelectSurvivor(m_a, m_b)  // Keep higher importance/newer
          MergeItems(survivor, Loser(m_a, m_b, survivor), layer)
      
      // PHASE 5: Contradiction resolution
      IF layer = Σ:
          contradictions ← DetectContradictions(layer)
          FOR each (σ_a, σ_b) IN contradictions:
              IF CanAutoResolve(σ_a, σ_b):
                  // Keep the one with more attestations and higher recency
                  ResolveContradiction(σ_a, σ_b, strategy="recency_and_attestation")
              ELSE:
                  ESCALATE_TO_HUMAN_REVIEW(σ_a, σ_b)
      
      EMIT Metrics("consolidation.pruned", pruned_count)
      EMIT Metrics("consolidation.merged", merged_count)
      EMIT Metrics("consolidation.layer_size", layer.count())
```

### 8.4 Merge Operation

When two items $m_a, m_b$ are merged:

$$m_{\text{merged}} = \left\langle \begin{array}{l} \text{content} = \text{LLM\_merge}(m_a.\text{content}, m_b.\text{content}) \\ \text{embedding} = \text{Embed}(m_{\text{merged}}.\text{content}) \\ \text{importance} = \max(m_a.\text{importance}, m_b.\text{importance}) \\ \text{access\_count} = m_a.\text{access\_count} + m_b.\text{access\_count} \\ \text{lineage} = m_a.\text{lineage} \cup m_b.\text{lineage} \\ \text{timestamp} = \max(m_a.\text{timestamp}, m_b.\text{timestamp}) \end{array} \right\rangle$$

---

## 9. Memory Write Policies and Admission Control

### 9.1 The Write Policy Framework

Every write to durable memory ($E$, $\Sigma$, $P$) must pass through a **policy gate**:

$$\phi_{\text{admit}}: \text{MemoryItem} \times \text{TargetLayer} \times \text{Context} \rightarrow \{\texttt{ACCEPT}, \texttt{REJECT}, \texttt{DEFER}, \texttt{MERGE}\}$$

### 9.2 Typed Write Policy Contract

```protobuf
message WritePolicy {
  string policy_id = 1;
  TargetLayer target = 2;
  
  // Admission gates (all must pass)
  float min_importance_score = 3;
  float max_similarity_to_existing = 4;    // dedup threshold
  float min_provenance_confidence = 5;
  bool require_factual_verification = 6;
  bool require_human_approval = 7;
  
  // Expiry
  google.protobuf.Duration default_ttl = 8;
  ExpiryStrategy expiry_strategy = 9;
  
  // Rate limiting
  uint32 max_writes_per_minute = 10;
  uint32 max_items_per_layer = 11;
  
  enum ExpiryStrategy {
    HARD_TTL = 0;           // Delete after TTL
    DECAY_SCORE = 1;        // Subject to pruning by importance decay
    NEVER_EXPIRE = 2;       // Permanent (semantic facts)
    REFRESH_ON_ACCESS = 3;  // TTL resets on retrieval
  }
}
```

### 9.3 Decision Matrix

| Condition | Working ($W$) | Session ($S$) | Episodic ($E$) | Semantic ($\Sigma$) | Procedural ($P$) |
|---|---|---|---|---|---|
| **Write Gate** | None (direct) | Schema check | Full admission | Promotion + contradiction | Trace alignment |
| **TTL** | Step-scoped | Session-scoped | Decay-based | Never / Annual review | Version-based |
| **Dedup** | N/A | Key-based | Embedding sim > 0.92 | NLI + embedding | Trigger sim > 0.90 |
| **Provenance Required** | No | Task ID | Full trace | Multi-episode attestation | Source episodes |
| **Human Approval** | No | No | On flag only | On contradiction | On low confidence |

---

## 10. End-to-End Memory Lifecycle: Integrated Agent Loop

### 10.1 Complete Memory Flow in One Agent Step

```
ALGORITHM 9: AgentStepWithMemory
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Input:
  user_input      — current user message or task trigger
  agent_state     — persistent agent state across steps
  M               — memory system ⟨W, S, E, Σ, P⟩

Output:
  response        — agent response
  updated_state   — updated agent state

PROCEDURE ExecuteAgentStep(user_input, agent_state, M):

  // ═══════════════════════════════════════════════════
  // PHASE 1: PLAN — Understand intent, decompose task
  // ═══════════════════════════════════════════════════
  1. task_analysis ← AnalyzeIntent(user_input, agent_state.history)
     subtasks ← Decompose(task_analysis)

  // ═══════════════════════════════════════════════════
  // PHASE 2: RETRIEVE — Pull relevant memory
  // ═══════════════════════════════════════════════════
  2. // Recall task state from session scratchpad
     task_context ← RecallFromScratchpad(agent_state.task_id,
                                           task_analysis.required_keys)
     
     // Retrieve from episodic, semantic, procedural memory
     retrieved_evidence ← IterativeRetrieve(
         query=task_analysis.retrieval_query,
         layers=[E, Σ],
         max_iterations=3,
         quality_threshold=0.8
     )
     
     // Retrieve applicable procedures
     applicable_procedures ← P.query(
         cos(trigger_embedding, Embed(task_analysis.description)) > 0.75
     )

  // ═══════════════════════════════════════════════════
  // PHASE 3: COMPILE — Build working memory (context)
  // ═══════════════════════════════════════════════════
  3. W_t ← PrefillCompiler.Compile(
         role_policy     = agent_state.role_policy,
         task_state      = task_context,
         retrieved       = retrieved_evidence,
         tools           = agent_state.available_tools,
         procedures      = applicable_procedures,
         history         = AdaptiveContextWindowManager(agent_state.history,
                                                         task_analysis, B_t),
         token_budget    = B_t
     )
     ASSERT tokens(W_t) ≤ N_max - R_reserved

  // ═══════════════════════════════════════════════════
  // PHASE 4: ACT — Generate response / execute tools
  // ═══════════════════════════════════════════════════
  4. llm_output ← LLM_generate(W_t)
     IF llm_output.contains_tool_calls:
         tool_results ← ExecuteTools(llm_output.tool_calls,
                                       authorization=agent_state.auth_scope)
         // Re-inject tool results and re-generate if needed
         W_t' ← InjectToolResults(W_t, tool_results)
         llm_output ← LLM_generate(W_t')

  // ═══════════════════════════════════════════════════
  // PHASE 5: VERIFY — Check output quality
  // ═══════════════════════════════════════════════════
  5. verification ← VerifyOutput(llm_output, task_analysis, retrieved_evidence)
     IF verification.hallucination_detected:
         // REPAIR: re-generate with explicit grounding constraint
         W_t_repair ← InjectGroundingConstraint(W_t, verification.flagged_claims)
         llm_output ← LLM_generate(W_t_repair)
         verification ← VerifyOutput(llm_output, task_analysis, retrieved_evidence)
     
     IF verification.quality_score < config.min_quality:
         ESCALATE("output_quality_below_threshold", llm_output, verification)

  // ═══════════════════════════════════════════════════
  // PHASE 6: COMMIT — Update memory layers
  // ═══════════════════════════════════════════════════
  6. response ← llm_output.final_response
     
     // Update session scratchpad with new task state
     new_task_state ← ExtractTaskState(llm_output, task_analysis)
     OffloadToScratchpad(agent_state.task_id, new_task_state)
     
     // Evaluate interaction for episodic storage
     episode_candidate ← ConstructEpisode(user_input, llm_output,
                                            tool_results, verification)
     admission_result ← EpisodicAdmissionController(episode_candidate, E, config)
     
     // Extract and store new user preferences / corrections
     preference_updates ← ExtractPreferences(user_input, llm_output)
     FOR each pref IN preference_updates:
         IF φ_admit(pref, target=Σ) = ACCEPT:
             WRITE pref TO Σ
     
     // Update history
     agent_state.history.append({user: user_input, assistant: response})
     
     // Emit observability
     EMIT Trace("agent.step", {
         task_id: agent_state.task_id,
         tokens_used: tokens(W_t),
         retrieval_count: |retrieved_evidence|,
         admission_result: admission_result,
         verification_score: verification.quality_score,
         latency_ms: elapsed()
     })

  7. RETURN (response, agent_state)
```

### 10.2 State Transition Diagram

```
┌────────────────────────────────────────────────────────────────────────┐
│                    MEMORY LIFECYCLE STATE MACHINE                      │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│   ┌──────────┐    context     ┌───────────┐                           │
│   │ EXTERNAL │──────load────→│  WORKING   │←───compile────┐          │
│   │  INPUT   │               │  MEMORY W  │               │          │
│   └──────────┘               └─────┬──────┘               │          │
│                                    │                       │          │
│                         ┌──────────┼──────────┐            │          │
│                         │ offload  │ retrieve │            │          │
│                         ▼          │          ▼            │          │
│                  ┌──────────┐      │   ┌──────────┐       │          │
│                  │ SESSION  │      │   │ EPISODIC │       │          │
│                  │    S     │      │   │    E     │       │          │
│                  └────┬─────┘      │   └────┬─────┘       │          │
│                       │            │        │              │          │
│            session    │            │        │ promote      │          │
│            end +      │            │        │ (≥N attested │          │
│            admit      │            │        │  episodes)   │          │
│                       ▼            │        ▼              │          │
│                  ┌──────────┐      │   ┌──────────┐       │          │
│                  │ EPISODIC │──────┘   │ SEMANTIC │───────┘          │
│                  │    E     │  recall  │    Σ     │  inject          │
│                  └────┬─────┘         └──────────┘                   │
│                       │                     ▲                         │
│            trace      │                     │ human validation       │
│            alignment  │                     │ or auto-resolve        │
│                       ▼                     │                         │
│                  ┌──────────┐               │                         │
│                  │PROCEDURAL│───────────────┘                         │
│                  │    P     │  successful procedure → fact            │
│                  └──────────┘                                         │
│                                                                        │
│   ┌─────────────────────────────────────────────────────────────┐     │
│   │  CONSOLIDATION AGENT (periodic background)                   │     │
│   │  • Prune stale/low-importance items                         │     │
│   │  • Merge near-duplicates                                     │     │
│   │  • Resolve contradictions                                    │     │
│   │  • Promote attested episodic → semantic                      │     │
│   │  • Extract recurring traces → procedural                     │     │
│   └─────────────────────────────────────────────────────────────┘     │
└────────────────────────────────────────────────────────────────────────┘
```

---

## 11. Production-Grade Reliability and Operational Concerns

### 11.1 Idempotent Memory Writes

Every write to durable memory must be **idempotent**. The idempotency key is derived from:

$$\text{IdempotencyKey}(m) = \text{SHA256}(m.\text{source.task\_id} \| m.\text{source.step\_index} \| m.\text{schema\_type})$$

Repeated writes with the same key are silently accepted (no-op) rather than creating duplicates.

### 11.2 Backpressure and Rate Limiting

Memory write throughput is bounded:

$$\text{WriteRate}_{\text{layer}} \leq \frac{R_{\max}^{\text{layer}}}{\Delta t} \quad \text{with jittered retry on rejection}$$

When the write queue exceeds capacity, apply **backpressure** by buffering in the session layer and deferring promotion.

### 11.3 Observability Contract

Every memory operation emits structured traces:

```protobuf
message MemoryTrace {
  string trace_id = 1;
  string operation = 2;    // "read", "write", "prune", "merge", "promote"
  string layer = 3;
  string item_id = 4;
  float latency_ms = 5;
  bool success = 6;
  string error_class = 7;
  uint32 tokens_involved = 8;
  map<string, string> metadata = 9;
}
```

Key metrics to monitor:

| Metric | Formula | Alert Threshold |
|--------|---------|-----------------|
| Memory Store Size | $|\mathcal{L}|$ per layer $\mathcal{L}$ | > 90% capacity |
| Retrieval Latency p99 | $P_{99}(\text{latency}_{\text{retrieve}})$ | > 500ms |
| Admission Rate | $\frac{|\text{accepted}|}{|\text{candidates}|}$ | < 5% or > 80% (miscalibrated) |
| Retrieval Hit Rate | $\frac{|\text{queries with } \geq 1 \text{ relevant result}|}{|\text{total queries}|}$ | < 60% |
| Contradiction Rate | $\frac{|\text{contradictions detected}|}{|\Sigma|}$ | > 2% |
| Context Budget Utilization | $\frac{|W_t|}{N_{\max} - R_{\text{reserved}}}$ | > 95% sustained |

### 11.4 Failure Recovery and Graceful Degradation

| Failure Mode | Detection | Recovery |
|---|---|---|
| Retrieval timeout | Deadline exceeded | Return partial results + flag `degraded_retrieval` |
| Memory store unavailable | Connection error / circuit breaker open | Fall back to working memory only; disable writes; queue for retry |
| Admission controller crash | Health check failure | Buffer candidates in session store; replay on recovery |
| Consolidation agent stall | Heartbeat timeout | Kill and restart; resume from last checkpoint |
| Embedding service down | gRPC error | Use cached embeddings; fall back to BM25-only retrieval |
| Token budget exceeded | Assert violation in compiler | Emergency truncation with priority-based eviction |

### 11.5 Cost Optimization

$$\text{Cost}_{\text{memory}} = \underbrace{C_{\text{embed}} \cdot N_{\text{embed}}}_{\text{embedding generation}} + \underbrace{C_{\text{store}} \cdot |\mathcal{M}|}_{\text{storage}} + \underbrace{C_{\text{retrieve}} \cdot N_{\text{queries}}}_{\text{retrieval ops}} + \underbrace{C_{\text{llm}} \cdot N_{\text{reflection}}}_{\text{LLM calls for scoring/reflection}}$$

Optimization strategies:

1. **Batch embedding generation**: Amortize embedding API calls by batching new items.
2. **Tiered storage**: Hot items in vector DB (HNSW index), warm in compressed store, cold in archive.
3. **Cache retrieval results**: LRU cache on $(query\_hash, layer, timestamp\_bucket) \rightarrow results$ with TTL.
4. **Lazy reflection**: Only invoke LLM-based importance scoring for items that pass cheap heuristic pre-filters.
5. **Shared embeddings**: Reuse task embeddings for both routing and retrieval scoring.

---

## 12. Evaluation Infrastructure for Memory Quality

### 12.1 Memory Quality Metrics

$$\text{MemoryQuality} = \frac{1}{|\mathcal{T}|} \sum_{\tau \in \mathcal{T}} \left[ w_{\text{ret}} \cdot \text{RetrievalPrecision}(\tau) + w_{\text{util}} \cdot \text{UtilizationRate}(\tau) + w_{\text{fresh}} \cdot \text{FreshnessPenalty}(\tau) \right]$$

where:

$$\text{RetrievalPrecision}(\tau) = \frac{|\text{retrieved items judged relevant by evaluator}|}{|\text{total retrieved items}|}$$

$$\text{UtilizationRate}(\tau) = \frac{|\text{retrieved items actually cited in response}|}{|\text{total retrieved items}|}$$

$$\text{FreshnessPenalty}(\tau) = 1 - \frac{|\text{retrieved items that are stale/outdated}|}{|\text{total retrieved items}|}$$

### 12.2 Automated Regression Testing

```
ALGORITHM 10: MemoryRegressionSuite
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PROCEDURE RunMemoryRegressions():

  // Test 1: Admission Consistency
  FOR each (candidate, expected_decision) IN admission_test_set:
      actual ← φ_admit(candidate, target_layer, context)
      ASSERT actual.decision = expected_decision
  
  // Test 2: Retrieval Quality
  FOR each (query, expected_items, min_recall) IN retrieval_test_set:
      results ← φ_retrieve(query, budget)
      recall ← |results ∩ expected_items| / |expected_items|
      ASSERT recall ≥ min_recall
  
  // Test 3: Deduplication Effectiveness
  INJECT known_duplicate INTO E
  RunConsolidation()
  ASSERT NOT EXISTS(known_duplicate, E)  // Should have been merged
  
  // Test 4: Contradiction Detection
  INJECT contradictory_fact INTO Σ
  contradictions ← DetectContradictions(Σ)
  ASSERT contradictory_fact IN contradictions
  
  // Test 5: Context Budget Compliance
  FOR each scenario IN budget_stress_test_set:
      W_t ← PrefillCompiler.Compile(scenario)
      ASSERT tokens(W_t) ≤ N_max - R_reserved
  
  // Test 6: Procedural Memory Accuracy
  FOR each (task_type, expected_procedure_steps) IN procedural_test_set:
      procedures ← P.query(task_type)
      step_accuracy ← SequenceAlignment(procedures[0].steps, expected_procedure_steps)
      ASSERT step_accuracy ≥ 0.85
```

---

## 13. Summary: Design Invariants for Production Memory Systems

| Invariant | Enforcement Mechanism |
|---|---|
| **No anonymous context**: every retrieved item carries provenance | `MemoryItem.provenance` is a required field; retrieval rejects items without it |
| **Hard token budget**: context window never exceeds $N_{\max} - R_{\text{reserved}}$ | `ASSERT` in `PrefillCompiler.Compile`; emergency truncation on violation |
| **Idempotent writes**: duplicate writes are no-ops | SHA256 idempotency key on every durable write |
| **Validated admission only**: no item enters durable memory without passing gates | `φ_admit` pipeline with schema, novelty, importance, provenance gates |
| **Bounded consolidation**: pruning never deletes actively-referenced items | Reference check before delete; soft-delete with archive |
| **Contradiction-safe semantics**: contradictions are detected and escalated | NLI-based contradiction detection; human escalation on ambiguity |
| **Observable memory health**: all operations emit structured traces and metrics | `MemoryTrace` protocol; dashboard alerts on anomalous rates |
| **Graceful degradation**: retrieval failure does not crash the agent | Timeout + partial result + fallback flag |
| **Cost-bounded operations**: LLM calls for scoring are rate-limited and cached | Lazy reflection; LRU cache on reflection outputs |
| **Separation of layers**: working ≠ session ≠ episodic ≠ semantic ≠ procedural | Typed stores with distinct schemas, TTLs, and write policies |

---

**This architecture transforms memory from passive storage into an actively managed, provenance-tracked, quality-gated, cost-bounded subsystem** that enables agents to retain what matters, discard what does not, retrieve with precision under latency constraints, and improve continuously through consolidation, promotion, and evaluation—at production scale.



# Memory Architecture for Agentic AI Systems: A Principal-Level Technical Report

---

## Abstract

This report presents a formal, production-grade treatment of memory architecture within agentic AI systems. Memory is formalized not as a passive data store but as a **bounded, layered, policy-governed control system** that mediates between an LLM's finite context window and the unbounded information space of persistent knowledge. We define the complete memory stack—working, session, episodic, semantic, and procedural layers—through typed interfaces, mathematical models, formal admission/eviction policies, provenance-tracked writes, and retrieval-optimized read paths. Every subsystem is specified with pseudoalgorithmic precision, mathematical underpinning, and explicit treatment of failure modes, token economics, latency constraints, and hallucination control.

---

## 1. Foundational Formalism: Memory as a Bounded Resource System

### 1.1 The Context Window as Constrained Working Memory

An LLM operates over a fixed token budget $B_{\text{ctx}}$. Following Karpathy's analogy—context window as RAM, model weights as CPU—we formalize the agent's **active reasoning capacity** as:

$$B_{\text{available}} = B_{\text{ctx}} - B_{\text{system}} - B_{\text{tools}} - B_{\text{reserved\_generation}}$$

where:

- $B_{\text{ctx}}$: Total context window (e.g., 128k tokens for GPT-4.1)
- $B_{\text{system}}$: Token cost of role policy, protocol bindings, and invariant instructions
- $B_{\text{tools}}$: Token cost of currently loaded tool schemas and affordances
- $B_{\text{reserved\_generation}}$: Minimum tokens reserved for model output generation

**Definition 1.1 (Memory Pressure).** The memory pressure $\mathcal{P}$ at time step $t$ is:

$$\mathcal{P}(t) = \frac{\sum_{i} \text{tokens}(m_i(t))}{B_{\text{available}}}$$

where $m_i(t)$ denotes the $i$-th memory item loaded into the active window at step $t$. When $\mathcal{P}(t) > \alpha_{\text{critical}}$ (empirically $\alpha_{\text{critical}} \approx 0.85$), reasoning quality degrades nonlinearly due to attention dilution across transformer layers.

**Theorem 1.1 (Context Saturation Degradation).** For a transformer with $L$ layers and attention head dimension $d_k$, the effective attention entropy $H_{\text{attn}}$ over $n$ context tokens satisfies:

$$H_{\text{attn}}(n) = -\sum_{j=1}^{n} \alpha_j \log \alpha_j, \quad \alpha_j = \frac{\exp(q \cdot k_j / \sqrt{d_k})}{\sum_{l=1}^{n} \exp(q \cdot k_l / \sqrt{d_k})}$$

As $n \to B_{\text{ctx}}$, the attention distribution flattens, $H_{\text{attn}}$ increases, and the model's capacity to discriminate high-signal tokens from noise decreases. This motivates aggressive context curation rather than naive context stuffing.

### 1.2 The Memory Hierarchy: Formal Layer Definitions

We define five distinct memory layers, each with typed contracts, admission policies, storage backends, and retrieval characteristics.

| Layer | Persistence | Latency | Write Policy | Storage Backend | Token Footprint |
|---|---|---|---|---|---|
| **Working Memory** | Ephemeral (task-scoped) | < 1ms (in-context) | Automatic | Context window itself | Direct consumption |
| **Session Memory** | Session-scoped | < 5ms | Append + compress | In-process buffer / Redis | Summarized injection |
| **Episodic Memory** | Persistent | < 50ms | Validated admission | Vector DB + metadata store | Retrieved on demand |
| **Semantic Memory** | Persistent | < 100ms | Curated ingestion | Knowledge graph + vector index | Retrieved on demand |
| **Procedural Memory** | Persistent | < 50ms | Workflow extraction | Structured plan store | Compiled into prefill |

**Definition 1.2 (Memory Layer Interface).** Each layer $\mathcal{L}_k$ exposes the following typed contract:

```
interface MemoryLayer<T> {
  read(query: Query, budget: TokenBudget, deadline: Timestamp) → Result<RankedItems<T>, MemoryError>
  write(item: T, provenance: Provenance, policy: WritePolicy) → Result<WriteReceipt, AdmissionError>
  evict(criteria: EvictionCriteria) → Result<EvictionReport, EvictionError>
  compact(strategy: CompactionStrategy) → Result<CompactionReport, CompactionError>
  inspect(filter: MetadataFilter) → Result<Paginated<T>, InspectionError>
}
```

---

## 2. Working Memory: The Active Reasoning Surface

### 2.1 Formal Definition

Working memory $\mathcal{W}(t)$ is the set of information items physically present in the LLM's context window at reasoning step $t$. It is **not a storage system**—it is the compiled runtime state assembled by the prefill compiler.

$$\mathcal{W}(t) = \Pi_{\text{sys}} \cup \Pi_{\text{task}}(t) \cup \mathcal{R}(t) \cup \mathcal{T}_{\text{loaded}}(t) \cup \mathcal{M}_{\text{injected}}(t) \cup \mathcal{H}_{\text{compressed}}(t)$$

where:

- $\Pi_{\text{sys}}$: System policy and role instructions (static)
- $\Pi_{\text{task}}(t)$: Current task objective, constraints, and decomposition state
- $\mathcal{R}(t)$: Retrieved evidence from episodic/semantic layers
- $\mathcal{T}_{\text{loaded}}(t)$: Tool schemas lazily loaded for the current step
- $\mathcal{M}_{\text{injected}}(t)$: Memory summaries promoted from session/episodic layers
- $\mathcal{H}_{\text{compressed}}(t)$: Compressed conversation history

### 2.2 Token Budget Allocation Algorithm

The prefill compiler allocates the available token budget across working memory components using a priority-weighted allocation scheme.

```
Algorithm 1: WorkingMemoryBudgetAllocation
────────────────────────────────────────────
Input:  B_available, component_set C = {c₁, ..., cₙ},
        priority weights w = {w₁, ..., wₙ},
        minimum allocations min = {min₁, ..., minₙ}
Output: allocation A = {a₁, ..., aₙ}

1.  B_remaining ← B_available
2.  // Phase 1: Satisfy hard minimums
3.  FOR each cᵢ in priority-descending order:
4.      aᵢ ← minᵢ
5.      B_remaining ← B_remaining - minᵢ
6.      IF B_remaining < 0:
7.          RAISE ContextBudgetExhausted(cᵢ)
8.  // Phase 2: Proportional allocation of surplus
9.  B_surplus ← B_remaining
10. W_total ← Σ wᵢ for all cᵢ where tokens(cᵢ) > minᵢ
11. FOR each cᵢ where tokens(cᵢ) > minᵢ:
12.     a_extra ← floor(B_surplus × wᵢ / W_total)
13.     aᵢ ← aᵢ + min(a_extra, tokens(cᵢ) - minᵢ)
14. // Phase 3: Redistribute unused allocation
15. B_unused ← B_available - Σ aᵢ
16. Allocate B_unused to generation reserve
17. RETURN A
```

**Priority ordering** (highest to lowest): System policy → Task state → Retrieved evidence → Tool schemas → Memory summaries → Compressed history.

### 2.3 Context Window Scratchpad: Task State Offloading

For multi-step tasks, intermediate state (partial plans, accumulated tool outputs, constraint satisfaction progress) must be managed without polluting the primary reasoning window.

**Definition 2.1 (Scratchpad).** A scratchpad $\mathcal{S}$ is an externalized key-value store with task-scoped lifetime:

$$\mathcal{S}: \text{TaskID} \times \text{StepID} \to \text{StructuredState}$$

```
Algorithm 2: ScratchpadOffloadAndRecall
────────────────────────────────────────
// OFFLOAD: After completing step k
1.  state_k ← extract_task_state(context_window)
2.  compressed_k ← compress(state_k, target_ratio=0.3)
3.  scratchpad.write(task_id, step_k, compressed_k, ttl=task_deadline)
4.  context_window.evict(state_k)  // Reclaim tokens

// RECALL: Before executing step k+1
5.  relevant_state ← scratchpad.read(task_id, steps=[k-2, k-1, k])
6.  decompressed ← decompress(relevant_state)
7.  context_window.inject(decompressed, priority=HIGH, budget=B_task)
```

The compression function uses **LLM-driven extractive summarization** with a strict token target:

$$\text{compress}(s, r) = \text{LLM}_{\text{summarizer}}\left(s, \; \text{constraint}: \frac{|\text{output}|}{|s|} \leq r\right)$$

---

## 3. Session Memory: Bounded Conversation State

### 3.1 Formal Model

Session memory $\mathcal{M}_{\text{sess}}$ captures the evolving dialogue and action history within a single interaction session. It bridges the gap between ephemeral working memory and persistent long-term stores.

**Definition 3.1 (Session).** A session $\sigma = (id, t_{\text{start}}, t_{\text{end}}, \mathcal{H}, \mathcal{D})$ where $\mathcal{H}$ is the ordered message history and $\mathcal{D}$ is the set of derived artifacts (summaries, extracted entities, preference signals).

### 3.2 Sliding Window with Progressive Summarization

Naive conversation history grows linearly. We apply a **hierarchical summarization** strategy that maintains semantic completeness while respecting token budgets.

$$\mathcal{H}_{\text{managed}}(t) = \mathcal{H}_{\text{recent}}(t) \cup \text{Summary}_{\text{mid}}(t) \cup \text{Summary}_{\text{old}}(t)$$

```
Algorithm 3: ProgressiveHistorySummarization
────────────────────────────────────────────────
Input:  History H = [m₁, m₂, ..., mₜ], budget B_history,
        window_recent=5, window_mid=20
Output: Managed history H_managed fitting within B_history

1.  H_recent ← H[t-window_recent : t]           // Full fidelity
2.  H_mid    ← H[t-window_mid : t-window_recent] // Summarize
3.  H_old    ← H[1 : t-window_mid]               // Deep compress

4.  IF tokens(H_recent) > B_history × 0.6:
5.      H_recent ← extractive_compress(H_recent, 0.6 × B_history)

6.  S_mid ← LLM_summarize(H_mid,
            instruction="Extract: decisions made, entities mentioned,
                         constraints established, open questions",
            budget=B_history × 0.25)

7.  S_old ← LLM_summarize(H_old ∪ prev_S_old,
            instruction="Retain only: user preferences, task outcomes,
                         corrections, unresolved items",
            budget=B_history × 0.15)

8.  H_managed ← S_old ⊕ S_mid ⊕ H_recent  // ⊕ = ordered concatenation
9.  ASSERT tokens(H_managed) ≤ B_history
10. RETURN H_managed
```

**Theorem 3.1 (Information Retention Bound).** Under progressive summarization with compression ratios $r_{\text{mid}}$ and $r_{\text{old}}$, the information retention over $T$ steps is:

$$I_{\text{retained}}(T) \geq I_{\text{recent}} + r_{\text{mid}} \cdot I_{\text{mid}} + r_{\text{old}}^{\lceil T / w_{\text{mid}} \rceil} \cdot I_{\text{old}}$$

where $I$ denotes mutual information between the original content and its compressed representation. The exponential decay in $I_{\text{old}}$ is acceptable because old information either promotes to episodic memory (if valuable) or is discarded (if not).

---

## 4. Long-Term Memory: Episodic, Semantic, and Procedural Stores

### 4.1 Episodic Memory: Validated Event Records

Episodic memory stores **specific interaction events**, outcomes, and corrections with full provenance.

**Definition 4.1 (Episodic Memory Item).**

```
type EpisodicItem = {
  id:           UUID
  timestamp:    ISO8601
  session_id:   SessionID
  event_type:   enum { CORRECTION, PREFERENCE, OUTCOME, FEEDBACK, EXCEPTION }
  content:      StructuredContent
  embedding:    Vector[d]
  provenance:   Provenance        // source, confidence, chain of custody
  importance:   float ∈ [0, 1]   // computed at admission
  access_count: uint
  last_accessed: ISO8601
  ttl:          Duration | null   // explicit expiry, null = permanent
  dedupe_hash:  SHA256            // for deduplication
}
```

#### 4.1.1 Admission Policy: The Importance Gate

Not every interaction merits persistence. The admission gate computes an importance score and applies a threshold.

**Definition 4.2 (Importance Score).** For a candidate memory item $m$, the importance score $\phi(m)$ is:

$$\phi(m) = \lambda_1 \cdot \text{novelty}(m) + \lambda_2 \cdot \text{correctional\_signal}(m) + \lambda_3 \cdot \text{user\_salience}(m) + \lambda_4 \cdot \text{task\_impact}(m)$$

where:

- $\text{novelty}(m) = 1 - \max_{m' \in \mathcal{E}} \text{sim}(\vec{m}, \vec{m'})$: Semantic distance from nearest existing memory
- $\text{correctional\_signal}(m) \in \{0, 1\}$: Whether $m$ represents a user correction or constraint that overrides prior behavior
- $\text{user\_salience}(m)$: LLM-judged importance on a calibrated [0,1] scale via structured output
- $\text{task\_impact}(m)$: Did this information materially change a task outcome? Binary or graded.
- $\lambda_1 + \lambda_2 + \lambda_3 + \lambda_4 = 1$, with $\lambda_2$ weighted highest (corrections are highest-value memories)

```
Algorithm 4: EpisodicMemoryAdmission
─────────────────────────────────────
Input:  Candidate item m, existing store E, threshold θ_admit
Output: ADMIT or REJECT with WriteReceipt or AdmissionError

1.  // Step 1: Deduplication
2.  hash ← SHA256(normalize(m.content))
3.  IF hash ∈ E.dedupe_index:
4.      existing ← E.get_by_hash(hash)
5.      existing.access_count += 1
6.      existing.last_accessed ← now()
7.      RETURN REJECT(reason="DUPLICATE", merged_with=existing.id)

8.  // Step 2: Near-duplicate detection (semantic)
9.  vec_m ← embed(m.content)
10. neighbors ← E.ann_search(vec_m, k=5, threshold=0.92)
11. IF |neighbors| > 0:
12.     // Merge rather than duplicate
13.     merged ← LLM_merge(m, neighbors[0], instruction="Combine,
             preserving provenance and latest corrections")
14.     E.update(neighbors[0].id, merged)
15.     RETURN REJECT(reason="NEAR_DUPLICATE_MERGED", merged_with=neighbors[0].id)

16. // Step 3: Importance scoring
17. novelty   ← 1 - max_similarity(vec_m, E)
18. correctional ← is_correction(m)  // Structural detection
19. salience  ← LLM_judge_importance(m, scale=[0,1], structured_output=true)
20. impact    ← assess_task_impact(m)
21. φ ← λ₁·novelty + λ₂·correctional + λ₃·salience + λ₄·impact

22. IF φ < θ_admit:
23.     RETURN REJECT(reason="BELOW_THRESHOLD", score=φ)

24. // Step 4: Provenance attachment
25. m.provenance ← {
        source: m.session_id,
        confidence: calibrated_confidence(m),
        chain: [agent_id, timestamp, admission_score=φ]
    }
26. m.embedding ← vec_m
27. m.importance ← φ
28. m.dedupe_hash ← hash
29. m.ttl ← compute_ttl(m.event_type)  // e.g., CORRECTION → null (permanent)

30. // Step 5: Validated write
31. E.insert(m, write_policy=APPEND_ONLY_WITH_AUDIT)
32. RETURN ADMIT(receipt={id: m.id, score: φ, stored_at: now()})
```

### 4.2 Semantic Memory: Curated Domain Knowledge

Semantic memory stores **general knowledge, facts, domain ontologies, and organizational policies**—information that is independent of specific interaction events.

**Definition 4.3 (Semantic Memory Item).**

```
type SemanticItem = {
  id:             UUID
  content:        StructuredContent
  content_type:   enum { FACT, POLICY, ONTOLOGY_EDGE, DOMAIN_RULE, DOCUMENT_CHUNK }
  embedding:      Vector[d]
  source:         SourceReference    // Document ID, URL, manual entry
  authority:      float ∈ [0, 1]    // Source trustworthiness
  freshness:      ISO8601           // Last verified/updated
  lineage:        List<TransformationStep>  // How this item was derived
  chunk_metadata: ChunkMetadata     // Parent doc, position, overlap
  tags:           Set<String>
  version:        SemanticVersion
}
```

#### 4.2.1 Document-Class-Aware Chunking

Different document types demand different chunking strategies. A single chunking approach produces suboptimal retrieval precision.

| Document Class | Chunking Strategy | Chunk Size | Overlap | Rationale |
|---|---|---|---|---|
| **Code** | AST-structural | Function/class boundaries | Import context | Preserves syntactic completeness |
| **Legal/Policy** | Hierarchical (section → clause) | Section-level with clause subchunks | Parent section header | Clause-level precision with section context |
| **Conversational logs** | Semantic (topic shift detection) | Topic segment | 2-turn overlap | Groups coherent dialogue segments |
| **Technical manuals** | Agentic (LLM-driven) | LLM-determined semantic units | Concept overlap | Maximizes conceptual completeness |
| **Tabular data** | Row/column schema-aware | Table + schema header per chunk | Schema always included | Preserves relational semantics |

```
Algorithm 5: AdaptiveChunking
─────────────────────────────
Input:  Document D, document_class c, target_chunk_tokens T_chunk
Output: List of chunks with metadata

1.  strategy ← CHUNKING_STRATEGY_MAP[c]
2.  SWITCH strategy:
3.    CASE STRUCTURAL:
4.      segments ← parse_structure(D, parser=AST|HTML|MARKDOWN)
5.      chunks ← split_at_boundaries(segments, max_tokens=T_chunk)
6.    CASE HIERARCHICAL:
7.      tree ← build_section_tree(D)
8.      chunks ← flatten_with_parent_context(tree, max_tokens=T_chunk)
9.    CASE SEMANTIC:
10.     embeddings ← sliding_window_embed(D, window=256, stride=128)
11.     breakpoints ← detect_topic_shifts(embeddings, threshold=θ_shift)
12.     chunks ← split_at_breakpoints(D, breakpoints, max_tokens=T_chunk)
13.   CASE AGENTIC:
14.     chunks ← LLM_chunk(D,
              instruction="Identify self-contained conceptual units.
                           Each chunk must be independently comprehensible.",
              target_size=T_chunk)
15. // Enrich all chunks
16. FOR each chunk c_i:
17.   c_i.embedding ← embed(c_i.content)
18.   c_i.metadata ← {parent_doc: D.id, position: i, total: |chunks|,
                       strategy: strategy, overlap_with: adjacent_ids}
19. RETURN chunks
```

### 4.3 Procedural Memory: Internalized Workflows

Procedural memory captures **learned operational sequences**—successful multi-step workflows, validated tool chains, and refined plans that the agent can reuse.

**Definition 4.4 (Procedural Memory Item).**

```
type ProceduralItem = {
  id:               UUID
  name:             String
  trigger_condition: StructuredPredicate   // When to invoke
  steps:            List<PlanStep>         // Ordered action sequence
  preconditions:    List<Constraint>
  postconditions:   List<Constraint>
  success_count:    uint                   // Reinforcement signal
  failure_count:    uint
  avg_latency:      Duration
  last_executed:    ISO8601
  provenance:       List<SessionID>        // Sessions that contributed
  version:          SemanticVersion
}
```

#### 4.3.1 Workflow Extraction and Reinforcement

```
Algorithm 6: ProceduralMemoryExtraction
───────────────────────────────────────
Input:  Completed task trace T = [(action₁, result₁), ..., (actionₙ, resultₙ)],
        task_outcome ∈ {SUCCESS, PARTIAL, FAILURE}
Output: Procedural memory update or new entry

1.  IF task_outcome = FAILURE:
2.      log_failure_trace(T)
3.      RETURN  // Do not extract failing workflows

4.  // Extract abstract plan from concrete trace
5.  abstract_plan ← LLM_abstract(T,
        instruction="Generalize this trace into a reusable workflow template.
                     Replace specific values with typed parameters.
                     Identify preconditions and postconditions.")

6.  // Check for existing similar procedure
7.  existing ← procedural_store.search(
        query=abstract_plan.trigger_condition,
        similarity_threshold=0.88)

8.  IF existing ≠ ∅:
9.      best_match ← existing[0]
10.     best_match.success_count += 1
11.     // Merge improvements
12.     IF abstract_plan has fewer steps OR lower estimated latency:
13.         best_match.steps ← merge_plans(best_match.steps, abstract_plan.steps)
14.         best_match.version ← increment(best_match.version)
15.     best_match.provenance.append(T.session_id)
16.     procedural_store.update(best_match)
17. ELSE:
18.     new_procedure ← ProceduralItem(
            name=LLM_name(abstract_plan),
            trigger_condition=abstract_plan.trigger,
            steps=abstract_plan.steps,
            preconditions=abstract_plan.preconditions,
            postconditions=abstract_plan.postconditions,
            success_count=1, failure_count=0,
            provenance=[T.session_id])
19.     procedural_store.insert(new_procedure)
```

**Reinforcement scoring** for procedure selection when multiple candidates match:

$$\text{score}(p) = \frac{p.\text{success\_count}}{p.\text{success\_count} + p.\text{failure\_count} + \epsilon} \cdot \exp\left(-\frac{t_{\text{now}} - p.\text{last\_executed}}{\tau}\right) \cdot \frac{1}{p.\text{avg\_latency} / \bar{L}}$$

where $\tau$ is a temporal decay constant and $\bar{L}$ is the mean latency across all procedures. This balances reliability, recency, and efficiency.

---

## 5. Retrieval Engine: Hybrid Multi-Signal Ranking

### 5.1 Query Decomposition and Routing

A single user query often spans multiple information needs. The retrieval engine must decompose and route subqueries before executing searches.

```
Algorithm 7: QueryDecompositionAndRouting
────────────────────────────────────────
Input:  User query Q, available memory layers L = {episodic, semantic, procedural}
Output: Ranked, provenance-tagged evidence set E

1.  // Decompose into subqueries
2.  subqueries ← LLM_decompose(Q,
        instruction="Break this query into independent information needs.
                     For each, specify: intent, expected_source_type,
                     temporal_scope, required_precision")
    // e.g., Q = "Book me a flight to Tokyo like last time"
    //   → sq₁: {intent: "retrieve past Tokyo booking preferences", source: episodic}
    //   → sq₂: {intent: "current flight options to Tokyo", source: tool/external}
    //   → sq₃: {intent: "booking workflow", source: procedural}

3.  // Route each subquery
4.  FOR each sq_i in subqueries IN PARALLEL:
5.      target_layers ← route(sq_i.source_type, sq_i.temporal_scope)
6.      FOR each layer in target_layers:
7.          results_i ← layer.read(
                query=sq_i,
                budget=per_subquery_token_budget,
                deadline=global_deadline - elapsed)
8.          tag_provenance(results_i, source=layer.name, query=sq_i)

9.  // Merge and deduplicate across subquery results
10. merged ← merge_results(all results_i)
11. deduplicated ← deduplicate(merged, similarity_threshold=0.90)

12. // Multi-signal ranking (see §5.2)
13. ranked ← hybrid_rank(deduplicated, Q)
14. E ← ranked[:top_k]  // Enforce retrieval budget
15. RETURN E
```

### 5.2 Hybrid Ranking Function

We combine multiple retrieval signals into a unified ranking score:

$$\text{score}(m, Q) = w_r \cdot \text{relevance}(m, Q) + w_a \cdot \text{authority}(m) + w_f \cdot \text{freshness}(m) + w_u \cdot \text{utility}(m) + w_p \cdot \text{provenance\_trust}(m)$$

where:

$$\text{relevance}(m, Q) = \alpha \cdot \text{BM25}(m, Q) + (1-\alpha) \cdot \cos(\vec{m}, \vec{Q})$$

This fuses **exact lexical match** (BM25) with **semantic similarity** (cosine over dense embeddings), addressing the well-known failure mode where pure semantic search misses exact terms and pure lexical search misses paraphrases.

$$\text{freshness}(m) = \exp\left(-\frac{t_{\text{now}} - m.\text{last\_updated}}{\tau_f}\right)$$

$$\text{authority}(m) = m.\text{source\_trust} \cdot \mathbb{1}[\text{provenance is verified}]$$

$$\text{utility}(m) = \frac{m.\text{access\_count\_recent}}{m.\text{access\_count\_total} + \epsilon} \cdot \text{task\_alignment}(m, \text{current\_task})$$

### 5.3 Cross-Encoder Reranking

After initial retrieval, a cross-encoder reranker provides a second-pass relevance assessment:

```
Algorithm 8: CrossEncoderReranking
──────────────────────────────────
Input:  Candidate set C (top-k₁ from hybrid retrieval), query Q, final_k
Output: Reranked top-k₂ results (k₂ < k₁)

1.  FOR each candidate c_i in C:
2.      // Cross-encoder processes (Q, c_i) jointly for deep interaction
3.      rerank_score_i ← cross_encoder.score(Q, c_i.content)
4.      c_i.final_score ← β · rerank_score_i + (1-β) · c_i.hybrid_score

5.  sorted ← sort(C, key=final_score, descending=true)
6.  RETURN sorted[:final_k]
```

The cross-encoder (e.g., a fine-tuned BERT or a small LLM used as a judge) evaluates the full interaction between query and document, capturing nuances that bi-encoder similarity misses. The interpolation parameter $\beta$ (typically 0.6–0.8) controls the weight of the reranker vs. the initial retrieval score.

### 5.4 Iterative Retrieval Refinement

When initial retrieval yields insufficient or ambiguous results:

```
Algorithm 9: IterativeRetrievalRefinement
────────────────────────────────────────
Input:  Query Q, max_iterations=3, quality_threshold θ_q
Output: Evidence set E

1.  E ← initial_retrieve(Q)
2.  quality ← LLM_judge_sufficiency(E, Q)  // Structured output: score ∈ [0,1]

3.  iteration ← 0
4.  WHILE quality < θ_q AND iteration < max_iterations:
5.      gaps ← LLM_identify_gaps(E, Q,
                instruction="What specific information is missing
                             or insufficiently supported?")
6.      refined_queries ← LLM_generate_refined_queries(gaps)
7.      E_new ← retrieve(refined_queries)
8.      E ← merge_and_deduplicate(E, E_new)
9.      E ← rerank(E, Q)  // Re-rank combined set
10.     quality ← LLM_judge_sufficiency(E, Q)
11.     iteration += 1

12. IF quality < θ_q:
13.     E.metadata.confidence = LOW
14.     E.metadata.gaps = gaps  // Explicit gap declaration

15. RETURN E
```

---

## 6. Memory Lifecycle Management: Eviction, Pruning, and Compaction

### 6.1 Eviction Policy: Composite Scoring

Memory items are periodically evaluated for eviction using a composite retention score:

$$\text{retain}(m) = \gamma_1 \cdot \text{recency}(m) + \gamma_2 \cdot \text{frequency}(m) + \gamma_3 \cdot \text{importance}(m) + \gamma_4 \cdot \text{correctional\_value}(m)$$

$$\text{recency}(m) = \exp\left(-\frac{t_{\text{now}} - m.\text{last\_accessed}}{\tau_r}\right)$$

$$\text{frequency}(m) = \frac{\log(1 + m.\text{access\_count})}{\log(1 + \max_j m_j.\text{access\_count}})}$$

**Hard eviction rules** (override scoring):

1. If $m.\text{ttl} \neq \text{null}$ and $t_{\text{now}} > m.\text{timestamp} + m.\text{ttl}$: **evict unconditionally**
2. If $m.\text{event\_type} = \text{CORRECTION}$: **never auto-evict** (requires explicit human override)
3. If $m$ has unresolved downstream references: **defer eviction** until references are resolved

```
Algorithm 10: PeriodicMemoryMaintenance
───────────────────────────────────────
Input:  Memory store E, capacity_limit C, target_utilization ρ_target
Output: Maintenance report

1.  // Phase 1: TTL-based expiry
2.  expired ← {m ∈ E : m.ttl ≠ null ∧ now() > m.timestamp + m.ttl}
3.  E.batch_evict(expired)

4.  // Phase 2: Score-based eviction if over capacity
5.  IF |E| > C × ρ_target:
6.      FOR each m in E:
7.          m.retain_score ← compute_retain(m)
8.      sorted ← sort(E, key=retain_score, ascending=true)
9.      evict_count ← |E| - floor(C × ρ_target)
10.     candidates ← sorted[:evict_count]
11.     // Filter out protected items
12.     candidates ← candidates.filter(m → m.event_type ≠ CORRECTION
                                         ∧ m.has_no_unresolved_references)
13.     E.batch_evict(candidates)

14. // Phase 3: Near-duplicate compaction
15. clusters ← cluster_by_embedding(E, threshold=0.93)
16. FOR each cluster with |cluster| > 1:
17.     merged ← LLM_merge_cluster(cluster,
              instruction="Merge into single canonical entry.
                           Preserve most recent data, all provenance chains,
                           and any correctional signals.")
18.     E.replace(cluster, merged)

19. // Phase 4: Embedding refresh for stale items
20. stale ← {m ∈ E : m.embedding_model_version < current_model_version}
21. FOR each m in stale IN BATCHES:
22.     m.embedding ← re_embed(m.content)

23. RETURN MaintenanceReport(expired, evicted, merged, re_embedded)
```

### 6.2 Memory Quality Invariants

The following invariants must hold continuously and are enforced via background validation agents:

| Invariant | Enforcement Mechanism |
|---|---|
| No duplicate content within $\delta_{\text{sim}} = 0.93$ semantic similarity | Admission gate + periodic compaction |
| Every item has non-null provenance | Schema validation at write time |
| Correctional items are never auto-evicted | Hard rule in eviction policy |
| No item exceeds its TTL | Periodic sweep + lazy check at read time |
| Embedding vectors match current model version | Background re-embedding job |
| Total store size $\leq C$ | Capacity-triggered eviction |

---

## 7. Context Pollution and Hallucination Control

### 7.1 The Context Pollution Problem

**Definition 7.1 (Context Pollution).** Context pollution occurs when low-quality, outdated, contradictory, or irrelevant memory items are injected into the working context, causing the LLM to generate responses that propagate or amplify errors.

The error propagation chain:

$$m_{\text{bad}} \xrightarrow{\text{retrieval}} \mathcal{W}(t) \xrightarrow{\text{generation}} \text{response}_t \xrightarrow{\text{feedback loop}} m'_{\text{bad}} \xrightarrow{\text{storage}} \mathcal{E}$$

A single bad memory item can compound across sessions if the erroneous output is itself stored as a new memory.

### 7.2 Multi-Gate Hallucination Prevention

```
Algorithm 11: MemoryInjectionQualityGate
─────────────────────────────────────────
Input:  Retrieved items R, current query Q, task context T
Output: Filtered, validated items R_safe

1.  R_safe ← []
2.  FOR each item r in R:
3.      // Gate 1: Provenance verification
4.      IF r.provenance.confidence < θ_provenance:
5.          SKIP r (log: "low provenance confidence")
6.      
7.      // Gate 2: Temporal validity
8.      IF r.freshness < θ_freshness AND r.content_type ∈ {FACT, POLICY}:
9.          flag r as "potentially stale"
10.         r.metadata.stale_warning = true
11.     
12.     // Gate 3: Contradiction detection
13.     contradictions ← detect_contradictions(r, R_safe, method="NLI_model")
14.     IF contradictions ≠ ∅:
15.         // Resolve by recency + authority
16.         winner ← resolve_contradiction(r, contradictions,
                       strategy="prefer_higher_authority_then_recency")
17.         IF winner ≠ r:
18.             SKIP r (log: "contradicted by higher-authority source")
19.     
20.     // Gate 4: Relevance threshold
21.     IF r.final_score < θ_relevance:
22.         SKIP r (log: "below relevance threshold")
23.     
24.     R_safe.append(r)

25. // Gate 5: Token budget enforcement
26. R_safe ← truncate_to_budget(R_safe, B_retrieval)
27. RETURN R_safe
```

### 7.3 Provenance-Tagged Injection

Every memory item injected into the context window carries explicit provenance markers so the LLM can assess source reliability:

```
[MEMORY: episodic | source=session_2024-12-15 | confidence=0.91 | age=3d]
User prefers window seat on flights longer than 4 hours.
Established via explicit user statement.
[/MEMORY]

[MEMORY: semantic | source=airline_policy_v2.3 | authority=0.95 | verified=2025-01-10]
ANA allows 2 checked bags on international economy bookings.
[/MEMORY]
```

This structured framing enables the LLM to appropriately weight and caveat information during generation, reducing hallucination from stale or low-confidence memories.

---

## 8. The Prefill Compiler: Memory-Aware Context Assembly

### 8.1 Architecture

The prefill compiler is the **central orchestration component** that assembles the complete context window as a deterministic, reproducible artifact.

```
Algorithm 12: PrefillCompilation
─────────────────────────────────
Input:  Task T, session S, agent config A, memory layers M,
        retrieval engine R, tool registry TR
Output: Compiled context window C_compiled

1.  // Phase 1: Static components
2.  C_sys ← load_system_policy(A.role, A.protocol_version)
3.  B_used ← tokens(C_sys)

4.  // Phase 2: Task state
5.  C_task ← compile_task_state(T.objective, T.constraints, T.decomposition)
6.  B_used += tokens(C_task)

7.  // Phase 3: Tool loading (lazy, task-relevant only)
8.  relevant_tools ← TR.discover(T.required_capabilities)
9.  C_tools ← compile_tool_schemas(relevant_tools, format=COMPACT)
10. B_used += tokens(C_tools)

11. // Phase 4: Memory retrieval and injection
12. B_memory ← floor((B_ctx - B_used - B_reserved_generation) × 0.4)
13. B_history ← floor((B_ctx - B_used - B_reserved_generation) × 0.3)
14. B_retrieval ← floor((B_ctx - B_used - B_reserved_generation) × 0.3)

15. // 4a: Session history (compressed)
16. C_history ← progressive_summarize(S.history, budget=B_history)  // Algo 3

17. // 4b: Episodic + semantic retrieval
18. evidence ← R.retrieve(T.objective, budget=B_retrieval)          // Algo 7-9
19. evidence ← quality_gate(evidence, T)                            // Algo 11
20. C_evidence ← format_with_provenance(evidence)

21. // 4c: Procedural memory (compiled into instructions)
22. procedures ← M.procedural.match(T.trigger_condition)
23. best_procedure ← select_by_reinforcement_score(procedures)      // §4.3.1
24. C_procedure ← compile_procedure_as_plan(best_procedure)

25. // 4d: Memory summaries (episodic patterns, user model)
26. C_memory_summary ← compile_user_model(M.episodic, user_id=S.user_id,
                         budget=B_memory)

27. // Phase 5: Assembly with budget enforcement
28. C_compiled ← ASSEMBLE(
        order=[C_sys, C_task, C_procedure, C_tools,
               C_memory_summary, C_evidence, C_history],
        total_budget=B_ctx - B_reserved_generation)

29. // Phase 6: Final validation
30. ASSERT tokens(C_compiled) ≤ B_ctx - B_reserved_generation
31. ASSERT no_contradictions(C_compiled)  // NLI check on injected items
32. log_compilation_manifest(C_compiled, trace_id=T.trace_id)

33. RETURN C_compiled
```

### 8.2 Compilation Determinism and Reproducibility

Every compilation produces a **manifest** containing:

```
type CompilationManifest = {
  trace_id:           TraceID
  timestamp:          ISO8601
  total_tokens:       uint
  budget_utilization:  float       // tokens_used / B_available
  component_breakdown: Map<ComponentName, TokenCount>
  retrieved_item_ids:  List<UUID>
  retrieval_scores:    List<float>
  procedure_id:        UUID | null
  history_compression_ratio: float
  quality_gate_rejections:   List<{item_id, reason}>
}
```

This manifest is stored alongside the agent trace for **observability, debugging, and replay**.

---

## 9. Production Reliability: Failure Modes and Mitigations

### 9.1 Failure Taxonomy

| Failure Mode | Impact | Mitigation |
|---|---|---|
| **Retrieval latency spike** | Context assembly exceeds deadline | Tiered deadlines per subquery; fallback to cached results; degrade gracefully by dropping lowest-priority memory layers |
| **Embedding service outage** | Cannot compute query/document embeddings | Circuit breaker pattern; fallback to BM25-only retrieval; cached embedding index serves reads |
| **Memory store corruption** | Invalid items enter working context | Write-ahead log; admission gate with schema validation; periodic integrity audit agent |
| **Token budget overflow** | Context exceeds model limit | Hard assertion in prefill compiler; progressive truncation with priority ordering |
| **Stale memory poisoning** | Outdated facts produce wrong answers | Freshness scoring in retrieval; explicit staleness warnings in provenance tags; TTL enforcement |
| **Contradictory memories** | Conflicting information confuses model | NLI-based contradiction detection at injection time; resolution by authority + recency |
| **Memory write amplification** | Storage grows unboundedly | Admission threshold; deduplication; periodic compaction; capacity-triggered eviction |

### 9.2 Idempotency and Retry Safety

All memory writes must be idempotent. The deduplication hash (SHA256 of normalized content) ensures that retried writes due to transient failures do not create duplicate entries.

```
write_idempotency_key = SHA256(normalize(content) || session_id || step_id)
IF store.exists(write_idempotency_key):
    RETURN existing_receipt  // Safe retry
ELSE:
    store.insert(item, idempotency_key=write_idempotency_key)
```

### 9.3 Backpressure and Rate Limiting

Under high load, the memory system applies backpressure:

$$\text{admit\_rate}(t) = \min\left(\text{rate\_limit}, \; \frac{C_{\text{remaining}}}{\Delta t_{\text{batch}}} \cdot (1 - \mathcal{P}_{\text{store}}(t))\right)$$

where $\mathcal{P}_{\text{store}}(t)$ is the current store utilization. When utilization approaches capacity, the admission rate drops, forcing the system to be more selective (higher $\theta_{\text{admit}}$) rather than accepting lower-quality memories.

---

## 10. Evaluation Infrastructure: Continuous Memory Quality Assurance

### 10.1 Memory System Metrics

| Metric | Definition | Target |
|---|---|---|
| **Retrieval Precision@k** | Fraction of top-k retrieved items judged relevant by human/LLM evaluator | ≥ 0.85 |
| **Retrieval Recall** | Fraction of relevant items in store that appear in retrieved set | ≥ 0.70 |
| **Admission Precision** | Fraction of admitted items that prove useful in future retrievals | ≥ 0.75 |
| **Context Pollution Rate** | Fraction of injected items that led to incorrect or degraded outputs | ≤ 0.05 |
| **Memory Staleness** | Fraction of items past freshness threshold at retrieval time | ≤ 0.10 |
| **Deduplication Efficiency** | 1 - (duplicates found in periodic audit / total items) | ≥ 0.98 |
| **Retrieval Latency P99** | 99th percentile retrieval latency | ≤ 100ms |
| **Compilation Latency P99** | 99th percentile prefill compilation latency | ≤ 500ms |

### 10.2 Automated Evaluation Pipeline

```
Algorithm 13: MemoryQualityCI
──────────────────────────────
// Runs on every deployment and on schedule

1.  // Test 1: Retrieval quality regression
2.  FOR each (query, expected_items) in retrieval_benchmark_set:
3.      retrieved ← retrieve(query, k=10)
4.      precision ← |retrieved ∩ expected_items| / k
5.      recall ← |retrieved ∩ expected_items| / |expected_items|
6.      ASSERT precision ≥ θ_precision AND recall ≥ θ_recall

7.  // Test 2: Admission gate correctness
8.  FOR each (item, expected_decision) in admission_test_set:
9.      decision ← admission_gate(item)
10.     ASSERT decision = expected_decision

11. // Test 3: Context pollution detection
12. FOR each (query, known_bad_items) in pollution_test_set:
13.     injected ← retrieve_and_gate(query)
14.     ASSERT known_bad_items ∩ injected = ∅

15. // Test 4: Eviction policy invariants
16. RUN maintenance_cycle()
17. ASSERT no item with event_type=CORRECTION was evicted
18. ASSERT no item past TTL remains in store
19. ASSERT store_size ≤ capacity_limit

20. // Test 5: End-to-end memory-augmented generation
21. FOR each (scenario, expected_behavior) in e2e_test_set:
22.     response ← agent.run(scenario)
23.     quality ← LLM_judge(response, expected_behavior)
24.     ASSERT quality ≥ θ_e2e
```

---

## 11. Operational Architecture: System Topology

```
┌─────────────────────────────────────────────────────────────────────┐
│                        AGENT RUNTIME                                │
│  ┌──────────────┐   ┌──────────────┐   ┌──────────────────────┐    │
│  │  PREFILL      │   │  AGENT LOOP  │   │  OBSERVATION LAYER   │    │
│  │  COMPILER     │◄──│  (Plan→Act→  │──►│  (Logs, Traces,      │    │
│  │  (Algorithm   │   │   Verify→    │   │   Metrics, UI State) │    │
│  │   12)         │   │   Repair)    │   │                      │    │
│  └──────┬───────┘   └──────┬───────┘   └──────────────────────┘    │
│         │                  │                                        │
│         ▼                  ▼                                        │
│  ┌──────────────────────────────────────┐                          │
│  │         MEMORY MANAGER               │                          │
│  │  ┌──────────┐  ┌──────────────────┐  │                          │
│  │  │ ADMISSION│  │  QUALITY GATE    │  │                          │
│  │  │ GATE     │  │  (Algorithm 11)  │  │                          │
│  │  │ (Algo 4) │  │                  │  │                          │
│  │  └────┬─────┘  └───────┬──────────┘  │                          │
│  └───────┼────────────────┼─────────────┘                          │
│          │                │                                         │
└──────────┼────────────────┼─────────────────────────────────────────┘
           │                │
           ▼                ▼
┌──────────────────────────────────────────────────────────────────────┐
│                     MEMORY INFRASTRUCTURE                            │
│                                                                      │
│  ┌───────────┐  ┌──────────┐  ┌────────────┐  ┌──────────────────┐  │
│  │ WORKING   │  │ SESSION  │  │ EPISODIC   │  │ SEMANTIC         │  │
│  │ (Context  │  │ (Redis/  │  │ (Vector DB │  │ (Vector DB +     │  │
│  │  Window)  │  │  Buffer) │  │ + Metadata)│  │  Knowledge Graph)│  │
│  └───────────┘  └──────────┘  └────────────┘  └──────────────────┘  │
│                                                                      │
│  ┌──────────────┐  ┌────────────────┐  ┌──────────────────────────┐ │
│  │ PROCEDURAL   │  │ RETRIEVAL      │  │ MAINTENANCE AGENT        │ │
│  │ (Plan Store) │  │ ENGINE         │  │ (Eviction, Compaction,   │ │
│  │              │  │ (Hybrid Rank + │  │  Audit, Re-embedding)    │ │
│  │              │  │  Rerank)       │  │  (Algorithm 10)          │ │
│  └──────────────┘  └────────────────┘  └──────────────────────────┘ │
│                                                                      │
│  ┌──────────────────────────────────────────────────────────────────┐│
│  │  EVAL CI: Retrieval benchmarks, admission tests, pollution      ││
│  │  detection, invariant checks (Algorithm 13)                     ││
│  └──────────────────────────────────────────────────────────────────┘│
└──────────────────────────────────────────────────────────────────────┘
```

---

## 12. Cost and Token Economics

### 12.1 Memory Cost Model

The total per-request memory cost is:

$$C_{\text{memory}}(t) = C_{\text{retrieval}}(t) + C_{\text{injection}}(t) + C_{\text{write}}(t)$$

$$C_{\text{retrieval}}(t) = n_{\text{subqueries}} \cdot (C_{\text{embed}} + C_{\text{search}} + C_{\text{rerank}})$$

$$C_{\text{injection}}(t) = \text{tokens\_injected}(t) \cdot P_{\text{input\_token}}$$

$$C_{\text{write}}(t) = \mathbb{1}[\text{admission}] \cdot (C_{\text{embed}} + C_{\text{importance\_judge}} + C_{\text{storage\_write}})$$

where $P_{\text{input\_token}}$ is the per-token input cost of the LLM, $C_{\text{embed}}$ is the embedding model cost, $C_{\text{importance\_judge}}$ is the cost of the LLM call used in the admission gate, and $C_{\text{search}}$ is the vector database query cost.

### 12.2 Optimization Strategies

| Strategy | Mechanism | Savings |
|---|---|---|
| **Embedding cache** | Cache embeddings for frequent queries with TTL | 40-60% reduction in embedding calls |
| **Lazy tool loading** | Load tool schemas only when task requires them | 10-30% context token savings |
| **Progressive summarization** | Compress old history instead of retaining verbatim | 50-70% history token reduction |
| **Retrieval budget caps** | Hard limit on retrieved items per request | Bounded injection cost |
| **Admission selectivity** | Higher $\theta_{\text{admit}}$ reduces write volume | Logarithmic storage growth vs. linear |
| **Batch compaction** | Periodic merge reduces store size | 15-25% storage reduction |

---

## 13. Trade-Off Analysis and Design Decisions

### 13.1 Precision vs. Recall in Retrieval

Agentic systems should **strongly prefer precision over recall** in memory retrieval. A missed relevant item produces a gap (which the LLM can sometimes compensate for via parametric knowledge), but an irrelevant or contradictory item injected into context produces active harm (context pollution, hallucination amplification). Therefore:

- Set retrieval top-k conservatively (k=5–10 rather than k=50)
- Apply quality gate as a hard filter after retrieval
- Accept recall loss in exchange for precision gain

### 13.2 Memory Write Selectivity vs. Coverage

Aggressive admission filtering ($\theta_{\text{admit}} \to 1$) prevents pollution but risks losing useful information. The optimal threshold is **task-dependent**:

- **Safety-critical domains** (medical, legal): $\theta_{\text{admit}} \geq 0.8$, with mandatory human validation for high-impact items
- **Conversational agents**: $\theta_{\text{admit}} \approx 0.5$, allowing more preferences to accumulate with periodic cleanup
- **Corrections and constraints**: Always admit regardless of threshold ($\lambda_2$ dominance in importance scoring ensures this)

### 13.3 Summarization Fidelity vs. Token Efficiency

Progressive summarization introduces information loss. The design explicitly accepts this trade-off:

- **Recent history** (last 5 turns): Full fidelity—no summarization
- **Mid-range** (5–25 turns): Extractive summarization preserving entities, decisions, and constraints
- **Old history** (25+ turns): Deep compression retaining only corrections, preferences, and unresolved items
- **Critical items** escape summarization entirely by promoting to episodic memory before they age out of the mid-range window

---

## 14. Conclusion

Memory in agentic AI systems is a **first-class architectural concern** requiring formal treatment across five distinct layers, each with typed contracts, admission policies, eviction strategies, and retrieval optimization. The system must enforce a hard boundary between ephemeral working context and durable persistent stores, with a prefill compiler that assembles the context window as a deterministic, budget-aware, provenance-tagged artifact.

The key architectural invariants are:

1. **Admit selectively**: Not every interaction deserves persistence. Importance scoring with correction-signal dominance prevents memory bloat while preserving high-value learning.
2. **Retrieve precisely**: Hybrid multi-signal retrieval with cross-encoder reranking and quality gates prevents context pollution—the primary vector for hallucination amplification.
3. **Evict systematically**: Composite retention scoring with hard rules for correctional items and TTL-based expiry maintains memory hygiene without losing critical knowledge.
4. **Compile deterministically**: The prefill compiler produces reproducible, budget-compliant context windows with full compilation manifests for observability and replay.
5. **Evaluate continuously**: Retrieval precision, admission correctness, pollution rate, and end-to-end generation quality are measured via automated CI pipelines, not anecdotal assessment.

Memory is what transforms a stateless text processor into a context-aware, self-improving agent. The engineering challenge is not storage—it is **curation, retrieval, and lifecycle management** under the hard constraints of finite context windows, latency budgets, cost targets, and correctness requirements.

---

*Report prepared following Principal Agentic AI Engineering standards. All algorithms are production-implementable. All mathematical formulations are operationally grounded. All trade-offs are explicitly declared.*